# DA3METRIC-LARGE — Ablation Study on KITTI Eigen-Split

**What this notebook does:**

| Step | Description |
|------|-------------|
| 1 | Define **8 loss function variants** (4 classic + 4 from recent research papers) |
| 2 | Define **3 architecture variants** (decoder-only, partial, full fine-tune) |
| 3 | Define **3 augmentation variants** |
| 4 | Define **3 optimizer/scheduler variants** |
| 5 | Run each ablation configuration for a fixed number of epochs |
| 6 | Evaluate on KITTI Eigen-Split test set with **all 4 scale variants** (unscaled / median / mean / affine) |
| 7 | Report metrics exactly as in `DA_3_metric_trend_analysis_FINAL.ipynb`: pixel-weighted true RMSE, MAE, AbsRel, SqRel, RMSE_log, log10, δ<1.25, Bias |
| 8 | Produce **15 analysis plots**: radar, bar, heatmap, learning curves, error-by-depth, scale distributions |
| 9 | **+5 new loss functions** | `masked_depth` (DA3 L_D), `mae_canonical` (DepthPro L_MAE), `eg_ssi` (UniDepthV2 L_EG-SSI), `tgm` (Video Depth Anything L_TGM), `da3_video_combined` (full recommended recipe) |
---

### Loss functions covered

| ID | Name | Paper |
|----|------|-------|
| `silog` | Scale-Invariant Log (SILog) | Eigen et al., NeurIPS 2014 |
| `berhu` | Reverse-Huber (BerHu) | Zwald & Lambert-Lacroix, 2012 |
| `combined_base` | SILog + BerHu + L1 + Grad (**original baseline**) | — |
| `vnl` | Virtual Normal Loss (VNL) | Yin et al., ICCV 2019 |
| `log_l1` | Log-L1 (AdaBins-style) | Bhat et al., CVPR 2021 |
| `ssim_l1` | SSIM + L1 (structure-aware) | Godard et al., CVPR 2017 |
| `affine_invariant` | Affine-Invariant SILog (MiDaS) | Ranftl et al., TPAMI 2022 |
| `robust_cauchy` | Cauchy (M-estimator) robust loss | Barron, CVPR 2019 |
| `masked_depth` | **DA3 L_D** | DA3, 2024 | NEW -- sparse LiDAR-aware |
| `mae_canonical` | **DepthPro L_MAE** | DepthPro, 2024 | NEW -- robust to noisy GT |
| `eg_ssi` | **UniDepthV2 L_EG-SSI** | UniDepthV2, 2024 | NEW -- RGB edge-guided |
| `tgm` | **Video DA L_TGM** | VideoDA, 2024 | NEW [sequential frames] |
| `da3_video_combined` | **Full recommended** | -- | NEW [sequential + all] |

---

### Architecture / Optimizer / Augmentation ablations also included


## 0 · Imports & Logging

In [1]:
# import os

# # 1. Force agreement on how GPUs are counted
# os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# # 2. Lock it to the very first GPU
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# import torch

# # 3. Safely wake up the CUDA context BEFORE checking memory
# if torch.cuda.is_available():
#     torch.cuda.init() 
#     print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB") 
# else:
#     print("PyTorch still cannot see the GPU!")

# from depth_anything_3.api import DepthAnything3

# try:
#     model = DepthAnything3.from_pretrained(
#         "depth-anything/DA3METRIC-LARGE",
#         cache_dir="/STORAGE/Arka/cache"
#     )
#     print("CPU load: OK, params:", sum(p.numel() for p in model.parameters())/1e6, "M")
#     model = model.to("cuda")
#     print("GPU move: OK")
# except Exception as e:
#     print(f"FAILED at: {type(e).__name__}: {e}")

In [2]:
# import os
# # Force Jupyter to see the GPU, overriding any weird kernel.json configs
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# import torch

# # Safely wake up the CUDA context BEFORE checking memory
# if torch.cuda.is_available():
#     torch.cuda.init() 
#     print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB") 
# else:
#     print("PyTorch still cannot see the GPU!")

# from depth_anything_3.api import DepthAnything3

# try:
#     model = DepthAnything3.from_pretrained(
#         "depth-anything/DA3METRIC-LARGE",
#         cache_dir="/STORAGE/Arka/cache"
#     )
#     print("CPU load: OK, params:", sum(p.numel() for p in model.parameters())/1e6, "M")
#     model = model.to("cuda")
#     print("GPU move: OK")
# except Exception as e:
#     print(f"FAILED at: {type(e).__name__}: {e}")

In [3]:
# import os
# # Force PyTorch to only look at the first GPU (Ordinal 0)
# # CRITICAL: This MUST be executed before importing torch
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# import torch
# print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB") 
# # /{torch.cuda.mem_get_info()[1]/1024**3:.1f} GB")

# from depth_anything_3.api import DepthAnything3
# try:
#     model = DepthAnything3.from_pretrained(
#         "depth-anything/DA3METRIC-LARGE",
#         cache_dir="/STORAGE/Arka/cache"
#     )
#     print("CPU load: OK, params:", sum(p.numel() for p in model.parameters())/1e6, "M")
#     model = model.to("cuda")
#     print("GPU move: OK")
# except Exception as e:
#     print(f"FAILED at: {type(e).__name__}: {e}")

In [4]:
# import os
# # Optional: force it to see only the first GPU
# # os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

# import torch

# print("PyTorch version:", torch.__version__)
# print("CUDA available:", torch.cuda.is_available())

# if torch.cuda.is_available():
#     print("Device count:", torch.cuda.device_count())
#     print("Current device:", torch.cuda.current_device())
#     print("Device name:", torch.cuda.get_device_name(0))
# else:
#     print("PyTorch cannot see any CUDA GPUs.")

In [5]:
from __future__ import annotations
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import copy
import csv
import glob
import json
import logging
import math
import shutil
import sys
import time
import warnings
from itertools import combinations
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from scipy.ndimage import uniform_filter1d
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
import torch.backends.cudnn as cudnn
cudnn.benchmark = True

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("ablation")
logging.getLogger("transformers").setLevel(logging.ERROR)

print("Imports OK  |  CUDA:", torch.cuda.is_available())


Imports OK  |  CUDA: True


## 1 · Configuration

> ✏️ Edit paths and ablation settings here before running.  
> `ABLATION_EPOCHS` controls how many epochs each variant runs — keep it small (2–5) for a quick sweep; increase to 10–20 for a thorough study.


In [6]:
# ── Paths ──────────────────────────────────────────────────────────────────────
KITTI_ROOT      = "/STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset_small"
TRAIN_ROOT      = os.path.join(KITTI_ROOT, "train")
TEST_ROOT       = os.path.join(KITTI_ROOT, "test")
CACHE_DIR       = "/STORAGE/Arka/cache"

OUTPUT_BASE     = "/STORAGE/Arka/a_MDE_Code/5_Fully_supervised_FT/super_ft_ablation_outputs"
CHECKPOINT_DIR  = os.path.join(OUTPUT_BASE, "checkpoints")
RESULTS_DIR     = os.path.join(OUTPUT_BASE, "results")
PLOT_DIR        = os.path.join(OUTPUT_BASE, "plots")
SUMMARY_CSV     = os.path.join(RESULTS_DIR, "ablation_summary.csv")

for d in [OUTPUT_BASE, CHECKPOINT_DIR, RESULTS_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID        = "depth-anything/DA3METRIC-LARGE"
ORTHO_NORM      = 300.0
FOCAL_LENGTH_X  = 721.5377   # KITTI default: avg(fx, fy) ≈ 721.5 px

# ── Camera / Focal-length source ──────────────────────────────────────────
#
# DA3METRIC-LARGE metric conversion (model card):
#   metric_depth = focal * net_output / 300.
# where `focal` = avg(fx, fy) in pixels.
#
# USE_MODEL_FOCAL = True  → use preds['intrinsics'] returned by the model.
#                           Falls back to FOCAL_LENGTH_X if unavailable.
# USE_MODEL_FOCAL = False → always use the hard-coded FOCAL_LENGTH_X.
#
USE_MODEL_FOCAL = False       # ← matches DA_3_metric_trend_analysis_FINAL
# DA3METRIC-LARGE doesn't have intrinsic prediction feature as other DA3 models

# ── Ablation training budget ──────────────────────────────────────────────────
ABLATION_EPOCHS = 5          # epochs per ablation run (use 10-20 for full study)
BATCH_SIZE      = 4           # reduced from 8 (use ACCUM_STEPS     = 2           # increased from 1 (effective batch = BATCH_SIZE * ACCUM_STEPS = 8) to keep effective batch=8)          # from 2
LEARNING_RATE   = 5e-5
WEIGHT_DECAY    = 1e-4
WARMUP_EPOCHS   = 1
MAX_GRAD_NORM   = 1.0
ACCUM_STEPS     = 2           # increased from 1 (effective batch = BATCH_SIZE * ACCUM_STEPS = 8)           # from 4
MIXED_PREC      = True
NUM_WORKERS     = 8          # from 8 # from 4
MAX_TRAIN       = 0          # 0 = all; set e.g. 500 for fast sweep
MAX_TEST        = 0

# ── Dataset ───────────────────────────────────────────────────────────────────
TARGET_H        = 378        # must be multiple of 14
TARGET_W        = 1246        # must be multiple of 14
MIN_DEPTH       = 1e-3
MAX_DEPTH       = 80.0

# ── Ablation dimensions to run ────────────────────────────────────────────────
# Set any to False to skip that ablation dimension
RUN_LOSS_ABLATION   = True
RUN_ARCH_ABLATION   = True
RUN_AUG_ABLATION    = True
RUN_OPTIM_ABLATION  = True

# ── Loss constants ────────────────────────────────────────────────────────────
EPS           = 1e-6
TGM_THRESHOLD = 0.05   # metres — masks dynamic pixels in L_TGM (Video Depth Anything, 2024)

print("Configuration loaded.")

# ── HPT & Early-Stopping ─────────────────────────────────────────────────────
# Hyperparameter search is done on the VALIDATION set (carved from TRAIN_ROOT).
# The EIGEN TEST SPLIT (TEST_ROOT) is touched exactly ONCE per ablation label.
#
# EARLY_STOP_PATIENCE  — stop training if val_rmse does not improve for N epochs
# VAL_SPLIT_FRAC       — fraction of TRAIN_ROOT samples reserved for validation
# TEMP_CKPT_DIR        — directory for temporary best-model checkpoints;
#                        files are deleted after test evaluation
# ─────────────────────────────────────────────────────────────────────────────
EARLY_STOP_PATIENCE  = 3          # patience epochs before early-stop triggers
VAL_SPLIT_FRAC       = 0.15       # 15 % of training frames → val set
VAL_SPLIT_SEED       = 42         # reproducible train/val split

TEMP_CKPT_DIR = os.path.join(OUTPUT_BASE, "temp_ckpts")
os.makedirs(TEMP_CKPT_DIR, exist_ok=True)

# Per-group HPT grids: list of dicts, each dict maps kwarg → value.
# Keys supported: "lr" (learning rate).
# Extend with more keys (e.g. "weight_decay", "lam") for finer sweeps.
HPT_LR_CANDIDATES  = [1e-4, 5e-5, 2e-5]          # swept for all groups

LOSS_HPT_GRID  = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
ARCH_HPT_GRID  = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
AUG_HPT_GRID   = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
OPTIM_HPT_GRID = [{"lr": lr} for lr in HPT_LR_CANDIDATES]

print("HPT / early-stop config loaded.")
print(f"  patience={EARLY_STOP_PATIENCE}  val_frac={VAL_SPLIT_FRAC}  "
      f"lr_candidates={HPT_LR_CANDIDATES}")


Configuration loaded.
HPT / early-stop config loaded.
  patience=3  val_frac=0.15  lr_candidates=[0.0001, 5e-05, 2e-05]


## 2 · Dataset Loader (KITTI Eigen-Split)

In [7]:
_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]

_train_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
])


def _read_kitti_depth(path: str) -> np.ndarray:
    raw = np.array(Image.open(path), dtype=np.uint16)
    return raw.astype(np.float32) / 256.0


def _parse_kitti_calib(date_dir: Path, drive_dir: Path) -> Optional[np.ndarray]:
    calib: Dict[str, np.ndarray] = {}
    for fname in ["calib_cam_to_cam.txt"]:
        for search in [date_dir, drive_dir]:
            p = search / fname
            if p.exists():
                for line in p.read_text().splitlines():
                    if ":" in line:
                        k, v = line.split(":", 1)
                        try:
                            calib[k.strip()] = np.array(v.split(), dtype=np.float64)
                        except Exception:
                            pass
    K = None
    try:
        if "P_rect_02" in calib:
            P = calib["P_rect_02"].reshape(3, 4)
            K = P[:3, :3].copy()
        elif "P2" in calib:
            P = calib["P2"].reshape(3, 4)
            K = P[:3, :3].copy()
    except Exception:
        pass
    return K


def _augment_base(rgb: np.ndarray, depth: np.ndarray,
                  rng: np.random.RandomState) -> Tuple[np.ndarray, np.ndarray]:
    """Horizontal flip + mild colour jitter (baseline augmentation)."""
    if rng.rand() < 0.5:
        rgb   = np.fliplr(rgb).copy()
        depth = np.fliplr(depth).copy()
    f  = rgb.astype(np.float32) / 255.0
    br = rng.uniform(0.8, 1.2)
    co = rng.uniform(0.8, 1.2)
    f  = np.clip((f - f.mean()) * co + f.mean() * br, 0.0, 1.0)
    return (f * 255.0).astype(np.uint8), depth


def _augment_flip_only(rgb: np.ndarray, depth: np.ndarray,
                       rng: np.random.RandomState) -> Tuple[np.ndarray, np.ndarray]:
    """Horizontal flip only — no colour jitter."""
    if rng.rand() < 0.5:
        rgb   = np.fliplr(rgb).copy()
        depth = np.fliplr(depth).copy()
    return rgb, depth


def _augment_full(rgb: np.ndarray, depth: np.ndarray,
                  rng: np.random.RandomState) -> Tuple[np.ndarray, np.ndarray]:
    """Flip + colour jitter + random crop/resize (strong augmentation)."""
    rgb, depth = _augment_base(rgb, depth, rng)
    # Random scale crop [0.85, 1.0]
    h, w = rgb.shape[:2]
    scale = rng.uniform(0.85, 1.0)
    new_h = int(h * scale)
    new_w = int(w * scale)
    y0 = rng.randint(0, h - new_h + 1)
    x0 = rng.randint(0, w - new_w + 1)
    rgb   = cv2.resize(rgb[y0:y0+new_h, x0:x0+new_w, :], (w, h), interpolation=cv2.INTER_LINEAR)
    depth = cv2.resize(depth[y0:y0+new_h, x0:x0+new_w],   (w, h), interpolation=cv2.INTER_NEAREST)
    return rgb, depth


AUG_FNS = {
    "none":  None,
    "flip":  _augment_flip_only,
    "base":  _augment_base,      # ← original baseline
    "full":  _augment_full,
}


class KITTIEigenDepthDataset(Dataset):
    def __init__(
        self,
        root: str,
        split: str = "train",
        target_hw: Tuple[int, int] = (TARGET_H, TARGET_W),
        min_depth: float = MIN_DEPTH,
        max_depth: float = MAX_DEPTH,
        augment: bool = True,
        aug_mode: str = "base",
        max_samples: int = 0,
        seed: int = 42,
    ):
        self.root       = Path(root)
        self.split      = split
        self.target_hw  = target_hw
        self.min_depth  = min_depth
        self.max_depth  = max_depth
        self.augment    = augment and (split == "train")
        self.aug_fn     = AUG_FNS.get(aug_mode)
        self.rng        = np.random.RandomState(seed)
        self._index: List[Dict] = []
        self._build_index()

        if max_samples > 0 and len(self._index) > max_samples:
            idx = self.rng.choice(len(self._index), max_samples, replace=False)
            self._index = [self._index[i] for i in sorted(idx)]

        log.info("[KITTI-%s] %d samples (aug=%s)", split, len(self._index), aug_mode)

    def _build_index(self):
        depth_glob = str(self.root / "*" / "*_sync" /
                         "proj_depth" / "groundtruth" / "image_02" / "*.png")
        depth_paths = sorted(glob.glob(depth_glob))
        if not depth_paths:
            depth_glob = str(self.root / "*" / "*" /
                             "proj_depth" / "groundtruth" / "image_02" / "*.png")
            depth_paths = sorted(glob.glob(depth_glob))

        for depth_path in depth_paths:
            parts = Path(depth_path).parts
            try:
                gt_idx    = [p for p in parts].index("proj_depth")
                drive_dir = Path(*parts[:gt_idx])
                date_dir  = drive_dir.parent
                fname     = parts[-1]
            except (ValueError, IndexError):
                continue
            rgb_path = drive_dir / "image_02" / "data" / fname
            if not rgb_path.exists():
                continue
            K = _parse_kitti_calib(date_dir, drive_dir)
            self._index.append(dict(
                rgb_path=str(rgb_path), depth_path=depth_path,
                K=K, sequence=drive_dir.name,
                frame_id=Path(fname).stem, date=date_dir.name,
            ))

    def __len__(self) -> int:
        return len(self._index)

    def __getitem__(self, idx: int) -> Dict:
        rec      = self._index[idx]
        rgb      = cv2.cvtColor(cv2.imread(rec["rgb_path"]), cv2.COLOR_BGR2RGB)
        depth_gt = _read_kitti_depth(rec["depth_path"])
        orig_hw  = rgb.shape[:2]
        H, W = self.target_hw
        rgb      = cv2.resize(rgb,      (W, H), interpolation=cv2.INTER_LINEAR)
        depth_gt = cv2.resize(depth_gt, (W, H), interpolation=cv2.INTER_NEAREST)
        if self.augment and self.aug_fn is not None:
            rgb, depth_gt = self.aug_fn(rgb, depth_gt, self.rng)
        depth_gt = np.clip(depth_gt, 0.0, self.max_depth)
        valid    = (depth_gt > self.min_depth) & (depth_gt < self.max_depth)
        K = rec["K"]
        if K is not None:
            K = K.copy()
            K[0] *= W / orig_hw[1]
            K[1] *= H / orig_hw[0]
        rgb_t   = _train_transform(Image.fromarray(rgb))
        depth_t = torch.from_numpy(depth_gt).unsqueeze(0).float()
        valid_t = torch.from_numpy(valid.astype(np.float32)).unsqueeze(0)
        K_t     = torch.from_numpy(K).float() if K is not None else torch.zeros(3, 3)
        has_K   = torch.tensor(K is not None, dtype=torch.bool)
        return {
            "rgb": rgb_t, "depth_gt": depth_t, "valid_mask": valid_t,
            "K": K_t, "has_K": has_K,
            "rgb_path": rec["rgb_path"], "depth_path": rec["depth_path"],
        }

# ---------------------------------------------------------------------------
# Sequential frame-pair dataset -- required for L_TGM / da3_video_combined
# ---------------------------------------------------------------------------
class KITTISequentialDepthDataset(Dataset):
    """
    Returns CONSECUTIVE frame pairs (frame_i, frame_{i+1}) from the same
    drive sequence, enabling the Temporal Gradient Matching loss (L_TGM).

    Engineering rationale
    ---------------------
    Standard KITTI Eigen-split loaders sample random static frames; they cannot
    provide the i->i+1 ordering that L_TGM requires.  This dataset:

      1. Groups all depth frames by (date, drive).
      2. Sorts each group by numeric frame_id (filename stem cast to int).
      3. Emits (frame_i, frame_{i+1}) pairs -- the final frame of every drive
         is excluded because it has no successor.
      4. Applies the SAME random seed augmentation to both frames so that the
         temporal depth-change signal is not corrupted by inconsistent flips.

    The returned dict has standard keys (rgb, depth_gt, valid_mask, K, has_K)
    for the current frame, plus next_rgb, next_depth_gt, next_valid_mask,
    next_K, next_has_K for the next frame.
    """

    def __init__(self, root, split="train",
                 target_hw=(TARGET_H, TARGET_W),
                 min_depth=MIN_DEPTH, max_depth=MAX_DEPTH,
                 aug_mode="base", max_samples=0, seed=42):
        self.root      = Path(root)
        self.split     = split
        self.target_hw = target_hw
        self.min_depth = min_depth
        self.max_depth = max_depth
        self.aug_fn    = AUG_FNS.get(aug_mode)
        self.augment   = (split == "train")
        self.rng       = np.random.RandomState(seed)
        self._pairs: List[Tuple[Dict, Dict]] = []
        self._build_pairs()
        if max_samples > 0 and len(self._pairs) > max_samples:
            idx = self.rng.choice(len(self._pairs), max_samples, replace=False)
            self._pairs = [self._pairs[i] for i in sorted(idx)]
        log.info("[KITTI-Sequential-%s] %d pairs (aug=%s)",
                 split, len(self._pairs), aug_mode)

    def _build_drive_index(self) -> Dict:
        """Returns {(date, drive): [records sorted by frame_id]}."""
        depth_glob = str(self.root / "*" / "*_sync" /
                         "proj_depth" / "groundtruth" / "image_02" / "*.png")
        depth_paths = sorted(glob.glob(depth_glob))
        if not depth_paths:
            depth_glob = str(self.root / "*" / "*" /
                             "proj_depth" / "groundtruth" / "image_02" / "*.png")
            depth_paths = sorted(glob.glob(depth_glob))

        by_drive: Dict = {}
        for dp in depth_paths:
            parts = Path(dp).parts
            try:
                gi        = list(parts).index("proj_depth")
                drive_dir = Path(*parts[:gi])
                date_dir  = drive_dir.parent
                fname     = parts[-1]
            except (ValueError, IndexError):
                continue
            rgb_path = drive_dir / "image_02" / "data" / fname
            if not rgb_path.exists():
                continue
            K   = _parse_kitti_calib(date_dir, drive_dir)
            key = (date_dir.name, drive_dir.name)
            rec = dict(rgb_path=str(rgb_path), depth_path=dp,
                       K=K, frame_id=int(Path(fname).stem))
            by_drive.setdefault(key, []).append(rec)

        for key in by_drive:
            by_drive[key].sort(key=lambda r: r["frame_id"])
        return by_drive

    def _build_pairs(self):
        for frames in self._build_drive_index().values():
            for i in range(len(frames) - 1):
                self._pairs.append((frames[i], frames[i + 1]))

    def _load_frame(self, rec: Dict) -> Dict:
        H, W    = self.target_hw
        rgb     = cv2.cvtColor(cv2.imread(rec["rgb_path"]), cv2.COLOR_BGR2RGB)
        depth   = _read_kitti_depth(rec["depth_path"])
        orig_hw = rgb.shape[:2]
        rgb     = cv2.resize(rgb,   (W, H), interpolation=cv2.INTER_LINEAR)
        depth   = cv2.resize(depth, (W, H), interpolation=cv2.INTER_NEAREST)
        depth   = np.clip(depth, 0.0, self.max_depth)
        valid   = (depth > self.min_depth) & (depth < self.max_depth)
        K = rec["K"]
        if K is not None:
            K = K.copy()
            K[0] *= W / orig_hw[1]
            K[1] *= H / orig_hw[0]
        return dict(rgb_np=rgb, depth=depth, valid=valid, K=K,
                    rgb_path=rec["rgb_path"], depth_path=rec["depth_path"])

    def _to_tensors(self, frame: Dict) -> Dict:
        rgb_t   = _train_transform(Image.fromarray(frame["rgb_np"]))
        depth_t = torch.from_numpy(frame["depth"]).unsqueeze(0).float()
        valid_t = torch.from_numpy(frame["valid"].astype(np.float32)).unsqueeze(0)
        K       = frame["K"]
        K_t     = torch.from_numpy(K).float() if K is not None else torch.zeros(3, 3)
        has_K   = torch.tensor(K is not None, dtype=torch.bool)
        return {"rgb": rgb_t, "depth_gt": depth_t, "valid_mask": valid_t,
                "K": K_t, "has_K": has_K,
                "rgb_path": frame["rgb_path"], "depth_path": frame["depth_path"]}

    def __len__(self):
        return len(self._pairs)

    def __getitem__(self, idx):
        rec_curr, rec_next = self._pairs[idx]
        curr = self._load_frame(rec_curr)
        nxt  = self._load_frame(rec_next)

        if self.augment and self.aug_fn is not None:
            # Apply IDENTICAL augmentation to both frames to preserve the
            # temporal depth-change signal (same seed => same flip/crop).
            aug_seed = int(self.rng.randint(0, 2**31))
            for frame in [curr, nxt]:
                rng_local = np.random.RandomState(aug_seed)
                frame["rgb_np"], frame["depth"] = self.aug_fn(
                    frame["rgb_np"], frame["depth"], rng_local)
                frame["valid"] = ((frame["depth"] > self.min_depth) &
                                  (frame["depth"] < self.max_depth))

        curr_t = self._to_tensors(curr)
        nxt_t  = self._to_tensors(nxt)
        result = dict(curr_t)
        result.update({f"next_{k}": v for k, v in nxt_t.items()})
        return result


def collate_fn(batch):
    from torch.utils.data.dataloader import default_collate
    tensor_keys = [k for k in batch[0] if isinstance(batch[0][k], torch.Tensor)]
    str_keys    = [k for k in batch[0] if isinstance(batch[0][k], str)]
    out = {k: default_collate([b[k] for b in batch]) for k in tensor_keys}
    out.update({k: [b[k] for b in batch] for k in str_keys})
    return out

collate_fn_sequential = collate_fn

print("Dataset helpers ready.")
print("  KITTIEigenDepthDataset        -- single-frame, standard ablation")
print("  KITTISequentialDepthDataset   -- consecutive pairs, required for L_TGM")


Dataset helpers ready.
  KITTIEigenDepthDataset        -- single-frame, standard ablation
  KITTISequentialDepthDataset   -- consecutive pairs, required for L_TGM


## 3 · DA3METRIC-LARGE Model Wrapper

In [8]:
_DA3_TRANSFORM = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
])


def load_da3_model(device: torch.device) -> nn.Module:
    """Load base model from HuggingFace Hub (always fresh for each ablation run)."""
    from depth_anything_3.api import DepthAnything3
    log.info("Loading %s ...", MODEL_ID)
    model = DepthAnything3.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR).to(device)
    #, torch_dtype=torch.float16
    #model = torch.compile(model, mode="default")
    return model


def _apply_arch(model: nn.Module, arch_mode: str) -> None:
    """
    Apply architecture variant by freezing/unfreezing parameters.

    arch_mode:
      'decoder_only'   → freeze encoder + backbone, train decoder & head only (original)
      'last2_blocks'   → only last 2 decoder blocks + head trainable
      'full'           → all parameters trainable (full fine-tune)
    """
    if arch_mode == "full":
        for p in model.parameters():
            p.requires_grad_(True)
        model.model.train()
        if hasattr(model.model, "gradient_checkpointing_enable"):
            model.model.gradient_checkpointing_enable()
        log.info("  [arch=full] All parameters trainable.")
        return

    if arch_mode == "decoder_only":
        for name, param in model.named_parameters():
            freeze = any(k in name for k in
                         ["encoder", "pretrained", "backbone", "net.net"])
            param.requires_grad_(not freeze)
        model.model.train()
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        log.info("  [arch=decoder_only] %.1fM params trainable", trainable/1e6)
        return

    if arch_mode == "last2_blocks":
        # Freeze everything first, then unfreeze last 2 decoder blocks and head
        for p in model.parameters():
            p.requires_grad_(False)
        # Identify and unfreeze last-2 decoder blocks by name
        decoder_block_names = [
            name for name, _ in model.named_parameters()
            if "decoder" in name.lower() or "head" in name.lower()
               or "depth_head" in name.lower() or "scratch" in name.lower()
        ]
        # Sort and take last 30% as proxy for "last 2 blocks"
        last_n = max(1, int(0.30 * len(decoder_block_names)))
        for name in decoder_block_names[-last_n:]:
            model.get_parameter(name).requires_grad_(True)
        model.model.train()
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        log.info("  [arch=last2_blocks] %.1fM params trainable", trainable/1e6)
        return

    raise ValueError(f"Unknown arch_mode: {arch_mode}")


def forward_batch(
    model: nn.Module,
    rgb_batch: torch.Tensor,
    K_batch:   torch.Tensor,
    has_K:     torch.Tensor,
) -> torch.Tensor:
    """Differentiable forward pass → metric depth [B, 1, H, W]."""
    B, C, H, W = rgb_batch.shape
    inp  = rgb_batch.unsqueeze(1)   # (B, 1, 3, H, W)
    preds = model.model(inp, export_feat_layers=[])

    if isinstance(preds, dict):
        raw = preds.get("depth", preds.get("metric_depth", list(preds.values())[0]))
        # Respect USE_MODEL_FOCAL: only use model intrinsics when the flag is set
        model_fx = None
        if USE_MODEL_FOCAL and "intrinsics" in preds and preds["intrinsics"] is not None:
            intr = preds["intrinsics"]  # [B,3,3] or [B,1,3,3]
            if intr.dim() == 4:
                intr = intr[:, 0]  # → [B,3,3]
            fx_m = intr[:, 0, 0]; fy_m = intr[:, 1, 1]
            avg_f = (fx_m + fy_m) / 2.0  # [B]
            valid = avg_f > 0
            if valid.all():
                model_fx = avg_f.view(B, 1, 1, 1)
        if model_fx is not None:
            fx_pred = model_fx
        else:
            fx_pred = torch.where(
                has_K.view(B, 1, 1, 1),
                K_batch[:, 0, 0].view(B, 1, 1, 1),
                torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                           device=rgb_batch.device, dtype=rgb_batch.dtype),
            )
    elif isinstance(preds, (tuple, list)):
        raw     = preds[0]
        fx_pred = torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                             device=rgb_batch.device, dtype=rgb_batch.dtype)
    else:
        raw     = preds
        fx_pred = torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                             device=rgb_batch.device, dtype=rgb_batch.dtype)

    if raw.dim() == 3:
        raw = raw.unsqueeze(1)
    depth = (fx_pred * raw) / ORTHO_NORM
    if depth.shape[-2:] != (H, W):
        depth = F.interpolate(depth, size=(H, W), mode="bilinear", align_corners=False)
    return depth


@torch.no_grad()
def run_da3_inference(
    model: nn.Module,
    image_np: np.ndarray,
    device: torch.device,
    fx: float = FOCAL_LENGTH_X,
) -> np.ndarray:
    """
    Run DA3METRIC-LARGE inference on one uint8 RGB image (H×W×3).

    Focal source is controlled by USE_MODEL_FOCAL (global config):
      True  → use preds['intrinsics'] avg(fx,fy) returned by the model;
              falls back to `fx` (FOCAL_LENGTH_X) if unavailable.
      False → always use the `fx` argument (FOCAL_LENGTH_X).
    """
    orig_h, orig_w = image_np.shape[:2]
    new_h = int(round(orig_h / 14) * 14)
    new_w = int(round(orig_w / 14) * 14)
    img_pil = Image.fromarray(image_np).resize((new_w, new_h), Image.Resampling.BICUBIC)
    t_img   = _DA3_TRANSFORM(img_pil)
    inp     = t_img.unsqueeze(0).unsqueeze(1).to(device)
    model.model.eval()
    preds = model(inp, export_feat_layers=[])
    if isinstance(preds, dict):
        raw = preds.get("depth", preds.get("metric_depth", list(preds.values())[0]))
    elif isinstance(preds, (tuple, list)):
        raw = preds[0]
    else:
        raw = preds

    # ── Determine focal length (mirrors run_da3() in trend analysis) ──────
    focal_used: float = fx   # default / fallback
    if USE_MODEL_FOCAL and isinstance(preds, dict):
        intrinsics = preds.get("intrinsics", None)
        if intrinsics is not None:
            # Handle [B,1,3,3] or [B,3,3] or [3,3]
            if intrinsics.dim() == 4:
                intrinsics = intrinsics[0, 0]   # → [3,3]
            elif intrinsics.dim() == 3:
                intrinsics = intrinsics[0]       # → [3,3]
            fx_m = float(intrinsics[0, 0])
            fy_m = float(intrinsics[1, 1])
            f_val = (fx_m + fy_m) / 2.0
            if f_val > 0:
                focal_used = f_val
                log.debug("run_da3_inference: model focal=%.2f px", focal_used)
        elif "focal" in preds:  # legacy scalar key
            raw_f = preds["focal"]
            f_val = float(raw_f.squeeze()) if isinstance(raw_f, torch.Tensor) else float(raw_f)
            if f_val > 0:
                focal_used = f_val

    if isinstance(raw, torch.Tensor):
        if raw.dim() == 3:
            raw = raw.unsqueeze(1)
        depth = (focal_used * raw) / ORTHO_NORM
        depth = F.interpolate(depth, size=(orig_h, orig_w),
                              mode="bilinear", align_corners=False)
    else:
        raise RuntimeError(f"Unexpected model output type {type(raw)}")
    return depth.squeeze().cpu().float().numpy()

print("Model helpers ready.")


Model helpers ready.


## 4 · Loss Functions (13 total — 8 original + 5 new)

Eight **original** loss functions plus **five new** losses from recent SOTA papers:

### Original losses (reimplemented for clarity)
- **SILog** — Scale-Invariant Log loss. Eigen et al., NeurIPS 2014.
- **BerHu** — Reverse Huber. Robust to outliers, quadratic for large errors.
- **CombinedBase** — SILog + BerHu + L1 + GradientLoss (the original baseline).

### New losses from research papers (ablation v2)
- **VNL** — Virtual Normal Loss. Yin et al., ICCV 2019.
- **Log-L1** — Log-space L1 used in AdaBins (Bhat et al., CVPR 2021).
- **SSIM + L1** — Structural similarity SSIM combined with L1. Godard et al., CVPR 2017.
- **Affine-Invariant SILog (MiDaS)** — Ranftl et al., TPAMI 2022.
- **Robust Cauchy (Barron 2019)** — Generalised robust loss.

### New losses from 2024 SOTA papers (+5 from DA3_Ablation_Study_v2)

| Loss | Rationale for dash-cam + LiDAR |
|------|-------------------------------|
| **`masked_depth`** (DA3 L_D) | Only supervised on valid LiDAR pixels; confidence-weighted to prioritise nearby reliable returns. |
| **`mae_canonical`** (DepthPro L_MAE) | MAE in inverse-depth space + multi-scale gradients; robust to noisy/sparse GT. |
| **`eg_ssi`** (UniDepthV2 L_EG-SSI) | Edge supervision via top-5% RGB gradient patches — bypasses LiDAR sparsity for boundary sharpness. |
| **`tgm`** (VideoDA L_TGM) | Temporal consistency without optical flow; static-object mask ignores moving cars/pedestrians. ⚠ Requires sequential loader. |
| **`da3_video_combined`** | Full recipe: `masked_depth + gradient + eg_ssi + tgm`. ⚠ Requires sequential loader. |

> **AVOID:** Gradient matching losses applied **directly to raw LiDAR** degrade training on sparse/noisy real-world GT.
> Use gradient losses only on pseudo-labels (as in `eg_ssi`) or on dense synthetic data.


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# ── Helper ───────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────

def _safe_log(x: torch.Tensor) -> torch.Tensor:
    return torch.log(x.clamp(min=EPS))


# ─────────────────────────────────────────────────────────────────────────────
# 1. SILog  (Eigen et al., NeurIPS 2014)
# ─────────────────────────────────────────────────────────────────────────────

class SILogLoss(nn.Module):
    """
    Scale-Invariant Log loss.
        L = sqrt( mean(d²) - λ * mean(d)² )
    where d_i = log(pred_i) - log(gt_i).

    Reference: Eigen, Puhrsch, Fergus. NeurIPS 2014.
    """
    def __init__(self, lam: float = 0.85):
        super().__init__()
        self.lam = lam

    def forward(self, pred, gt, mask):
        pred = pred.squeeze(1); gt = gt.squeeze(1); mask = mask.squeeze(1).bool()
        if not mask.any():
            return pred.sum() * 0.0
        d = _safe_log(pred[mask]) - _safe_log(gt[mask])
        return torch.sqrt((d**2).mean() - self.lam * d.mean()**2 + EPS)


# ─────────────────────────────────────────────────────────────────────────────
# 2. BerHu  (Zwald & Lambert-Lacroix, 2012)
# ─────────────────────────────────────────────────────────────────────────────

class BerHuLoss(nn.Module):
    """
    Reverse Huber loss. L1 for small errors, L2 for large ones.
    Threshold c is set adaptively as fraction of the max absolute error.

    Reference: Laina et al., 3DV 2016 (popularised in MDE).
    """
    def __init__(self, threshold: float = 0.20):
        super().__init__()
        self.c = threshold

    def forward(self, pred, gt, mask):
        pred = pred.squeeze(1); gt = gt.squeeze(1); mask = mask.squeeze(1).bool()
        if not mask.any():
            return pred.sum() * 0.0
        diff = (pred - gt).abs()[mask]
        c    = (self.c * diff.max().detach()).clamp(min=1e-6)
        l1   = diff[diff <= c]
        l2   = diff[diff >  c]
        loss = l1.sum()
        if l2.numel() > 0:
            loss = loss + ((l2**2 + c**2) / (2 * c)).sum()
        return loss / diff.numel()


# ─────────────────────────────────────────────────────────────────────────────
# 3. GradientLoss  (image-gradient matching)
# ─────────────────────────────────────────────────────────────────────────────

class GradientLoss(nn.Module):
    """
    Penalises horizontal and vertical gradient differences between
    predicted and GT depth, encouraging edge sharpness.
    """
    def forward(self, pred, gt, mask):
        def _grad(x):
            dx = x[:, :, :, :-1] - x[:, :, :, 1:]
            dy = x[:, :, :-1, :] - x[:, :, 1:, :]
            return dx, dy
        m = mask.float()
        p_dx, p_dy = _grad(pred * m)
        g_dx, g_dy = _grad(gt   * m)
        mx_dx = m[:, :, :, :-1] * m[:, :, :, 1:]
        mx_dy = m[:, :, :-1, :] * m[:, :, 1:, :]
        return (((p_dx - g_dx).abs() * mx_dx).mean() +
                ((p_dy - g_dy).abs() * mx_dy).mean())


# ─────────────────────────────────────────────────────────────────────────────
# 4. Combined Baseline  (SILog + BerHu + L1 + Gradient)
# ─────────────────────────────────────────────────────────────────────────────

class CombinedBaseLoss(nn.Module):
    """
    Original baseline loss: weighted combination of SILog + BerHu + L1 + Grad.
    Weights: w_silog=1.0, w_berhu=1.0, w_l1=0.5, w_grad=0.1.
    """
    def __init__(self, w_silog=1.0, w_berhu=1.0, w_l1=0.5, w_grad=0.1):
        super().__init__()
        self.w_silog = w_silog; self.w_berhu = w_berhu
        self.w_l1 = w_l1;      self.w_grad = w_grad
        self.silog = SILogLoss(); self.berhu = BerHuLoss()
        self.grad  = GradientLoss()

    def forward(self, pred, gt, mask):
        pred_c = pred.clamp(EPS, MAX_DEPTH)
        l_si = self.silog(pred_c, gt, mask)
        l_bh = self.berhu(pred_c, gt, mask)
        l_l1 = (((pred_c - gt).abs()) * mask).sum() / (mask.sum() + EPS)
        l_gr = self.grad(pred_c, gt, mask)
        total = self.w_silog*l_si + self.w_berhu*l_bh + self.w_l1*l_l1 + self.w_grad*l_gr
        return total, {"total": total, "silog": l_si, "berhu": l_bh, "l1": l_l1, "grad": l_gr}


# ─────────────────────────────────────────────────────────────────────────────
# 5. VNL — Virtual Normal Loss  (Yin et al., ICCV 2019)
# ─────────────────────────────────────────────────────────────────────────────

class VNLLoss(nn.Module):
    """
    Virtual Normal Loss.

    Randomly samples N triplets of pixels. For each triplet (i, j, k),
    it computes the virtual normal of the plane they span in 3-D camera
    space using the GT depth and the predicted depth.  The loss penalises
    the cosine distance between the two sets of normals, enforcing
    geometric/planarity consistency that SILog and L1 cannot capture.

    Reference:
        Yin, Liu, Shen, Chen. "Enforcing geometric constraints of virtual
        normal for depth prediction." ICCV 2019.
        https://arxiv.org/abs/1907.12209
    """
    def __init__(self, num_triplets: int = 2000, focal: float = FOCAL_LENGTH_X):
        super().__init__()
        self.num_triplets = num_triplets
        self.focal = focal

    def _depth_to_3d(self, depth: torch.Tensor) -> torch.Tensor:
        """Backproject depth map [B,1,H,W] to 3-D points [B,H*W,3]."""
        B, _, H, W = depth.shape
        u = torch.arange(W, dtype=depth.dtype, device=depth.device).view(1, 1, W).expand(B, H, W)
        v = torch.arange(H, dtype=depth.dtype, device=depth.device).view(1, H, 1).expand(B, H, W)
        cx = W / 2.0; cy = H / 2.0
        x = (u - cx) * depth.squeeze(1) / self.focal
        y = (v - cy) * depth.squeeze(1) / self.focal
        z = depth.squeeze(1)
        pts = torch.stack([x, y, z], dim=-1)          # [B, H, W, 3]
        return pts.view(B, H * W, 3)                   # [B, N_pts, 3]

    def forward(self, pred, gt, mask):
        pred_c = pred.clamp(EPS, MAX_DEPTH)
        B, _, H, W = pred_c.shape
        mask_flat = mask.squeeze(1).view(B, -1).bool()  # [B, H*W]

        # Sample triplets from valid pixels
        pts_pred = self._depth_to_3d(pred_c)  # [B, N, 3]
        pts_gt   = self._depth_to_3d(gt)      # [B, N, 3]

        total_loss = torch.tensor(0.0, device=pred.device)
        n_valid = 0

        for b in range(B):
            valid_idx = mask_flat[b].nonzero(as_tuple=False).squeeze(1)
            if valid_idx.numel() < 3:
                continue
            n_sample = min(self.num_triplets * 3, valid_idx.numel())
            perm     = torch.randperm(valid_idx.numel(), device=pred.device)[:n_sample]
            idx      = valid_idx[perm]
            # Form triplets
            n_tri = len(idx) // 3
            if n_tri == 0:
                continue
            i_idx = idx[:n_tri]
            j_idx = idx[n_tri:2*n_tri]
            k_idx = idx[2*n_tri:3*n_tri]

            p_p = pts_pred[b]; g_p = pts_gt[b]
            # Vectors for normals
            v1_p = p_p[j_idx] - p_p[i_idx]
            v2_p = p_p[k_idx] - p_p[i_idx]
            v1_g = g_p[j_idx] - g_p[i_idx]
            v2_g = g_p[k_idx] - g_p[i_idx]
            # Cross product → normal
            n_p  = torch.cross(v1_p, v2_p, dim=-1)   # [N_tri, 3]
            n_g  = torch.cross(v1_g, v2_g, dim=-1)
            # Normalise
            n_p  = F.normalize(n_p + EPS, dim=-1)
            n_g  = F.normalize(n_g + EPS, dim=-1)
            # Cosine loss
            cos_sim = (n_p * n_g).sum(dim=-1).clamp(-1 + 1e-6, 1 - 1e-6)
            total_loss = total_loss + (1.0 - cos_sim).mean()
            n_valid += 1

        if n_valid == 0:
            return pred.sum() * 0.0, {"total": pred.sum() * 0.0}
        total_loss = total_loss / n_valid
        return total_loss, {"total": total_loss}


# ─────────────────────────────────────────────────────────────────────────────
# 6. Log-L1 + Gradient  (AdaBins-style, Bhat et al., CVPR 2021)
# ─────────────────────────────────────────────────────────────────────────────

class LogL1Loss(nn.Module):
    """
    Log-space L1 (|log pred − log gt|) combined with gradient regularisation.
    Used in AdaBins to suppress large errors at close and distant ranges.

    Reference:
        Bhat, Alhashim, Wonka. "AdaBins: Depth Estimation Using Adaptive
        Discretization." CVPR 2021. https://arxiv.org/abs/2011.14141
    """
    def __init__(self, w_grad: float = 0.1):
        super().__init__()
        self.w_grad = w_grad
        self.grad   = GradientLoss()

    def forward(self, pred, gt, mask):
        pred_c = pred.clamp(EPS, MAX_DEPTH)
        log_pred = _safe_log(pred_c)
        log_gt   = _safe_log(gt.clamp(EPS))
        l1_log   = ((log_pred - log_gt).abs() * mask).sum() / (mask.sum() + EPS)
        l_grad   = self.grad(pred_c, gt, mask)
        total    = l1_log + self.w_grad * l_grad
        return total, {"total": total, "log_l1": l1_log, "grad": l_grad}


# ─────────────────────────────────────────────────────────────────────────────
# 7. SSIM + L1  (Godard et al., CVPR 2017 — Monodepth)
# ─────────────────────────────────────────────────────────────────────────────

def _ssim_loss(pred: torch.Tensor, gt: torch.Tensor,
               mask: torch.Tensor, window_size: int = 7,
               C1: float = 1e-4, C2: float = 9e-4) -> torch.Tensor:
    """
    Structural similarity loss — SSIM-based photometric error adapted for depth.
    A gaussian-weighted 7×7 window computes local mean and variance.
    """
    pad = window_size // 2
    kernel = torch.ones(1, 1, window_size, window_size,
                        device=pred.device, dtype=pred.dtype) / (window_size ** 2)

    def local_mean(x):
        return F.conv2d(x, kernel, padding=pad)

    mu_p  = local_mean(pred)
    mu_g  = local_mean(gt)
    mu_pp = local_mean(pred * pred)
    mu_gg = local_mean(gt   * gt)
    mu_pg = local_mean(pred * gt)

    sigma_p  = (mu_pp - mu_p * mu_p).clamp(min=0)
    sigma_g  = (mu_gg - mu_g * mu_g).clamp(min=0)
    sigma_pg = mu_pg - mu_p * mu_g

    ssim_map = ((2 * mu_p * mu_g + C1) * (2 * sigma_pg + C2)) / (
                (mu_p**2 + mu_g**2 + C1) * (sigma_p + sigma_g + C2))
    loss_map = (1 - ssim_map) / 2.0      # [0, 1]
    return (loss_map * mask).sum() / (mask.sum() + EPS)


class SSIMPlusL1Loss(nn.Module):
    """
    alpha * SSIM + (1-alpha) * L1 depth supervision.
    Structural similarity adds texture/boundary coherence beyond pixel errors.

    Reference:
        Godard, Mac Aodha, Brostow. "Unsupervised Monocular Depth Estimation
        with Left-Right Consistency." CVPR 2017.
        https://arxiv.org/abs/1609.09082
    """
    def __init__(self, alpha: float = 0.85):
        super().__init__()
        self.alpha = alpha

    def forward(self, pred, gt, mask):
        pred_c  = pred.clamp(EPS, MAX_DEPTH)
        # Normalise to [0,1] range for SSIM to work robustly
        scale   = MAX_DEPTH
        l_ssim  = _ssim_loss(pred_c / scale, gt.clamp(EPS) / scale, mask)
        l_l1    = ((pred_c - gt).abs() * mask).sum() / (mask.sum() + EPS)
        total   = self.alpha * l_ssim + (1 - self.alpha) * l_l1
        return total, {"total": total, "ssim": l_ssim, "l1": l_l1}


# ─────────────────────────────────────────────────────────────────────────────
# 8. Affine-Invariant SILog  (MiDaS / DPT — Ranftl et al., TPAMI 2022)
# ─────────────────────────────────────────────────────────────────────────────

class AffineInvariantSILogLoss(nn.Module):
    """
    Affine-Invariant Scale-Shift-Invariant SILog (MiDaS loss).

    Before computing SILog, the prediction is aligned to the GT via OLS:
        pred_aligned = s * pred + t
    where (s, t) = argmin || s*pred + t - gt ||² over valid pixels.

    This makes the loss insensitive to global scale/shift ambiguity — ideal
    when the model has not yet been calibrated to metric depth, or when mixing
    datasets.  Includes gradient regularisation for sharpness.

    Reference:
        Ranftl, Bochkovskiy, Koltun. "Vision Transformers for Dense Prediction."
        TPAMI 2022 (MiDaS / DPT). https://arxiv.org/abs/2103.13413
    """
    def __init__(self, lam: float = 0.85, w_grad: float = 0.1, w_metric: float = 0.3):
        super().__init__()
        self.lam      = lam
        self.w_grad   = w_grad
        self.w_metric = w_metric   # FIX: metric anchor weight — prevents raw-scale drift
        self.silog    = SILogLoss(lam=lam)
        self.silog_metric = SILogLoss(lam=0.0)  # FIX: λ=0 → pure L2 in log-space, anchors scale
        self.grad     = GradientLoss()

    # def _align(self, pred: torch.Tensor, gt: torch.Tensor,
    #            mask: torch.Tensor) -> torch.Tensor:
    #     """OLS align: returns s*pred + t per batch item."""
    #     B = pred.shape[0]
    #     aligned = pred.clone()
    #     for b in range(B):
    #         m   = mask[b, 0].bool()
    #         p_v = pred[b, 0][m].float()
    #         g_v = gt[b, 0][m].float().clamp(EPS)
    #         if p_v.numel() < 2:
    #             continue
    #         A   = torch.stack([p_v, torch.ones_like(p_v)], dim=1)   # [N, 2]
    #         # Least-squares via normal equations
    #         try:
    #             params, _, _, _ = torch.linalg.lstsq(A, g_v.unsqueeze(1))
    #             s = params[0, 0]; t = params[1, 0]
    #             aligned[b, 0] = (s * pred[b, 0] + t).clamp(EPS, MAX_DEPTH)
    #         except Exception:
    #             pass
    #     return aligned
    def _align(self, pred: torch.Tensor, gt: torch.Tensor,
               mask: torch.Tensor) -> torch.Tensor:
        """Memory-efficient OLS align: returns s*pred + t per batch item."""
        B = pred.shape[0]
        aligned = pred.clone()
        for b in range(B):
            m   = mask[b, 0].bool()
            p_v = pred[b, 0][m].float()
            g_v = gt[b, 0][m].float().clamp(EPS)
            if p_v.numel() < 2:
                continue
            # Memory-efficient analytical OLS
            p_mean = p_v.mean()
            g_mean = g_v.mean()
            p_var = ((p_v - p_mean) ** 2).mean()
            # Prevent division by zero if predictions are completely flat
            if p_var < 1e-6:
                continue 
            cov = ((p_v - p_mean) * (g_v - g_mean)).mean()
            s = cov / p_var
            t = g_mean - s * p_mean
            aligned[b, 0] = (s * pred[b, 0] + t).clamp(EPS, MAX_DEPTH)
            
        return aligned

    def forward(self, pred, gt, mask):
        pred_c   = pred.clamp(EPS, MAX_DEPTH)
        pred_ali = self._align(pred_c, gt, mask)
        l_si     = self.silog(pred_ali, gt, mask)
        l_gr     = self.grad(pred_ali, gt, mask)
        # FIX: metric anchor — penalises raw (unaligned) scale so the network
        #      cannot freely drift while the affine term absorbs all gradient.
        #      Uses λ=0 SILog (pure log-L2) on the unaligned pred_c so it
        #      teaches absolute metric scale without competing with l_si.
        l_metric = self.silog_metric(pred_c, gt, mask)
        total    = l_si + self.w_grad * l_gr + self.w_metric * l_metric
        return total, {"total": total, "silog": l_si, "grad": l_gr, "metric_anchor": l_metric}


# ─────────────────────────────────────────────────────────────────────────────
# 9. Robust Cauchy Loss  (Barron, CVPR 2019)
# ─────────────────────────────────────────────────────────────────────────────

class RobustCauchyLoss(nn.Module):
    """
    Generalised robust loss from Barron (CVPR 2019).
    At alpha=0: Cauchy loss (very robust, heavy-tailed).
    At alpha=2: L2 loss.
    Combined with SILog for metric calibration.

        L(x; alpha, c) = |alpha - 2| / alpha * ((x/c)^2 / |alpha - 2| + 1)^(alpha/2) - 1

    For alpha=0 (Cauchy): L(x) = log(0.5 * (x/c)^2 + 1)

    The c (scale) parameter is set adaptively per batch as the median residual.

    Reference:
        Barron. "A General and Adaptive Robust Loss Function." CVPR 2019.
        https://arxiv.org/abs/1701.03077
    """
    def __init__(self, alpha: float = 0.0, w_silog: float = 0.5):
        super().__init__()
        self.alpha   = alpha
        self.w_silog = w_silog
        self.silog   = SILogLoss()

    def forward(self, pred, gt, mask):
        pred_c = pred.clamp(EPS, MAX_DEPTH)
        mask_b = mask.bool()
        if not mask_b.any():
            return pred.sum() * 0.0, {"total": pred.sum() * 0.0}

        residuals = (pred_c - gt)[mask_b]
        c = residuals.abs().median().detach().clamp(min=1e-3)  # adaptive scale

        alpha = self.alpha
        if abs(alpha) < 1e-6:   # Cauchy
            rob_loss = torch.log(0.5 * (residuals / c) ** 2 + 1).mean()
        elif abs(alpha - 2.0) < 1e-6:  # L2
            rob_loss = (0.5 * (residuals / c) ** 2).mean()
        else:
            rob_loss = (abs(alpha - 2) / alpha) * (
                ((residuals / c) ** 2 / abs(alpha - 2) + 1) ** (alpha / 2) - 1
            ).mean()

        l_si  = self.silog(pred_c, gt, mask)
        total = rob_loss + self.w_silog * l_si
        return total, {"total": total, "cauchy": rob_loss, "silog": l_si}


# ─────────────────────────────────────────────────────────────────────────────
# Loss registry
# ─────────────────────────────────────────────────────────────────────────────



# =============================================================================
# NEW LOSSES (+5 from DA3_Ablation_Study_v2) 
# =============================================================================

# =============================================================================
# 10. MaskedDepthLoss -- L_D from Depth Anything 3
# =============================================================================
class MaskedDepthLoss(nn.Module):
    """
    Sparse-LiDAR-aware masked depth loss derived from DA3's L_D.

    Only valid LiDAR pixels are supervised (sparse-GT safe).
    A confidence weight D_{c,p} = 1/(gt+eps) prioritises nearby, more
    reliable LiDAR returns. The lambda_c log-term regularises the confidence
    and prevents collapse:

        L_D = (1/Z) * sum_p  m_p * (D_c * |pred - gt| - lam_c * log(D_c))

    Reference: Depth Anything 3, Sec 3.2.
    """

    def __init__(self, lambda_c: float = 0.5):
        super().__init__()
        self.lambda_c = lambda_c

    def forward(self, pred, gt, mask):
        p = pred.clamp(EPS, MAX_DEPTH)
        if not mask.bool().any():
            return p.sum() * 0., {"total": p.sum() * 0.}
        D_c      = 1.0 / gt.clamp(min=EPS)
        D_c_norm = D_c / (D_c[mask.bool()].mean().detach() + EPS)
        per_px   = (D_c_norm * (p - gt).abs()
                    - self.lambda_c * torch.log(D_c_norm.clamp(EPS)))
        loss     = (per_px * mask.float()).sum() / (mask.float().sum() + EPS)
        return loss, {"total": loss}


# =============================================================================
# 11. MAECanonicalLoss -- Depth Pro (Bochkovskiy et al., 2024)
# =============================================================================
class MAECanonicalLoss(nn.Module):
    """
    Depth Pro's primary real-world training loss.

    Converts depths to canonical inverse depth c = 1/d (emphasising nearby
    objects where LiDAR is densest) and applies masked MAE in that space.
    Multi-scale gradient matching (L_MAGE over n_scales) drives the network
    toward sharp depth boundaries.

    Specifically chosen for robustness to corrupted/sparse real-world GT.

    Reference: "Depth Pro: Sharp Monocular Metric Depth in Less Than a
    Second." arXiv 2024.
    """

    def __init__(self, w_grad: float = 0.5, n_scales: int = 4):
        super().__init__()
        self.w_grad   = w_grad
        self.n_scales = n_scales
        self.grad     = GradientLoss()

    def forward(self, pred, gt, mask):
        p  = pred.clamp(EPS, MAX_DEPTH)
        g  = gt.clamp(EPS, MAX_DEPTH)
        ip = 1.0 / p
        ig = 1.0 / g

        l_mae = ((ip - ig).abs() * mask.float()).sum() / (mask.float().sum() + EPS)

        l_grad_total = torch.tensor(0., device=pred.device)
        pc, gc, mc = ip, ig, mask.float()
        for s in range(self.n_scales):
            l_grad_total = l_grad_total + self.grad(pc, gc, mc.gt(0.5).float())
            if s < self.n_scales - 1:
                pc = F.avg_pool2d(pc, 2, stride=2)
                gc = F.avg_pool2d(gc, 2, stride=2)
                mc = F.avg_pool2d(mc, 2, stride=2)
        l_grad_total = l_grad_total / self.n_scales

        t = l_mae + self.w_grad * l_grad_total
        return t, {"total": t, "mae_inv": l_mae, "ms_grad": l_grad_total}


# =============================================================================
# 12. EdgeGuidedSSILoss -- L_EG-SSI  (UniDepthV2, Piccinelli et al., 2024)
# =============================================================================
class EdgeGuidedSSILoss(nn.Module):
    """
    Edge-Guided Scale-Shift Invariant Loss from UniDepthV2.

    Aligns depth boundaries to visual image boundaries WITHOUT relying on
    sparse LiDAR for edge supervision:

      1. Compute Sobel gradient magnitude on the RGB image.
      2. Select pixels in the top `top_pct` (5%) of gradient magnitude.
      3. Extract local patches (patch_size x patch_size) at those locations.
      4. Standardise: N_w(D_w) = (D_w - median) / (MAD + eps).
      5. L = L1 between standardised pred and GT patches.

    Highly effective when depth GT is coarse/sparse (LiDAR).

    Reference: Piccinelli et al. "UniDepthV2." arXiv 2024.
    """

    def __init__(self, top_pct: float = 0.05,
                 patch_size: int = 16, n_patches: int = 64):
        super().__init__()
        self.top_pct    = top_pct
        self.patch_size = patch_size
        self.n_patches  = n_patches
        kx = torch.tensor([[-1., 0., 1.],
                            [-2., 0., 2.],
                            [-1., 0., 1.]]).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.],
                            [ 0.,  0.,  0.],
                            [ 1.,  2.,  1.]]).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    def _grad_mag(self, rgb: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb[:, 0:1] + 0.587 * rgb[:, 1:2] + 0.114 * rgb[:, 2:3]
        gx   = F.conv2d(gray, self.kx, padding=1)
        gy   = F.conv2d(gray, self.ky, padding=1)
        return (gx ** 2 + gy ** 2).sqrt()

    @staticmethod
    def _norm_patch(patch: torch.Tensor) -> torch.Tensor:
        flat = patch.reshape(-1)
        med  = flat.median()
        mad  = (flat - med).abs().mean().clamp(min=1e-6)
        return (patch - med) / mad

    def forward(self, pred, gt, mask, rgb=None):
        """rgb [B,3,H,W] used for edge detection; falls back to GT grad if None."""
        p   = pred.clamp(EPS, MAX_DEPTH)
        B, _, H, W = p.shape
        ps2 = self.patch_size // 2

        if rgb is not None:
            gm = self._grad_mag(rgb)
        else:
            gx = gt[:, :, :, 1:] - gt[:, :, :, :-1]
            gy = gt[:, :, 1:, :] - gt[:, :, :-1, :]
            gm = F.pad(gx.abs(), (0, 1, 0, 0)) + F.pad(gy.abs(), (0, 0, 0, 1))

        total   = torch.tensor(0., device=pred.device)
        n_terms = 0

        for b in range(B):
            gm_b = gm[b, 0]
            thr  = torch.quantile(gm_b.flatten(), 1.0 - self.top_pct)
            locs = (gm_b >= thr).nonzero(as_tuple=False)
            ok   = ((locs[:, 0] >= ps2) & (locs[:, 0] < H - ps2) &
                    (locs[:, 1] >= ps2) & (locs[:, 1] < W - ps2))
            locs = locs[ok]
            if locs.numel() == 0:
                continue
            perm = torch.randperm(len(locs), device=pred.device)[:self.n_patches]
            locs = locs[perm]
            for loc in locs:
                y, x = int(loc[0]), int(loc[1])
                pp   = p[b, 0, y-ps2:y+ps2, x-ps2:x+ps2]
                gp   = gt[b, 0, y-ps2:y+ps2, x-ps2:x+ps2]
                mp   = mask[b, 0, y-ps2:y+ps2, x-ps2:x+ps2]
                if mp.float().mean() < 0.25:
                    continue
                total   = total + (self._norm_patch(pp) - self._norm_patch(gp)).abs().mean()
                n_terms += 1

        if n_terms == 0:
            return pred.sum() * 0., {"total": pred.sum() * 0.}
        t = total / n_terms
        return t, {"total": t}


# =============================================================================
# 13. TemporalGradientMatchingLoss -- L_TGM (Video Depth Anything, 2024)
# =============================================================================
class TemporalGradientMatchingLoss(nn.Module):
    """
    L_TGM from Video Depth Anything -- temporal consistency WITHOUT optical flow.

    Core idea: the depth change at the same pixel location between frame i and
    frame i+1 should match the change in ground truth.

        L_TGM = (1/(N-1)) * sum_i  || |d_{i+1}-d_i| - |g_{i+1}-g_i| ||_1

    Computed ONLY where |g_{i+1}-g_i| < tgm_threshold (default 0.05 m).
    This masking purposefully IGNORES dynamic objects (moving cars, pedestrians)
    that violate rigid-scene temporal assumptions -- crucial for dash-cam data.

    Combined with a spatial SILog term (beta=1.0), weighted alpha=10.0.

    WARNING: Requires KITTISequentialDepthDataset (consecutive frame pairs).
    The loop in Section 9 automatically selects the correct loader.
    Degrades gracefully to pure SILog when only single frames are available.

    Reference: Shao et al. "Video Depth Anything." arXiv 2024.
    """

    def __init__(self, tgm_threshold: float = TGM_THRESHOLD, w_ssi: float = 1.0):
        super().__init__()
        self.threshold = tgm_threshold
        self.w_ssi     = w_ssi
        self.silog     = SILogLoss()

    def forward(self, pred_curr, pred_next, gt_curr, gt_next,
                mask_curr, mask_next):
        joint_mask = (mask_curr.bool() & mask_next.bool()).float()
        if joint_mask.sum() < 1:
            l = self.silog(pred_curr.clamp(EPS, MAX_DEPTH), gt_curr, mask_curr)
            return l, {"total": l,
                       "tgm": torch.tensor(0., device=pred_curr.device),
                       "ssi": l}

        pc = pred_curr.clamp(EPS, MAX_DEPTH)
        pn = pred_next.clamp(EPS, MAX_DEPTH)
        d_pred  = (pn - pc).abs()
        d_gt    = (gt_next - gt_curr).abs()
        static_mask = (d_gt < self.threshold).float() * joint_mask

        if static_mask.sum() < 1:
            l = self.silog(pc, gt_curr, mask_curr)
            return l, {"total": l,
                       "tgm": torch.tensor(0., device=pc.device),
                       "ssi": l}

        l_tgm = ((d_pred - d_gt).abs() * static_mask).sum() / (static_mask.sum() + EPS)
        l_ssi = self.silog(pc, gt_curr, mask_curr)
        t     = 10.0 * l_tgm + self.w_ssi * l_ssi   # weights from VDA paper
        return t, {"total": t, "tgm": l_tgm, "ssi": l_ssi}

    def forward_single(self, pred, gt, mask):
        """Graceful fallback when sequential data is unavailable."""
        l = self.silog(pred.clamp(EPS, MAX_DEPTH), gt, mask)
        return l, {"total": l}


# =============================================================================
# 14. DA3VideoCombinedLoss -- Full recommended recipe
# =============================================================================
class DA3VideoCombinedLoss(nn.Module):
    """
    Full recommended combination for fine-tuning DA3METRIC-LARGE on dash-cam
    videos with sparse LiDAR GT:

        L = w_masked * L_D        (sparse-LiDAR robust, confidence-weighted)
          + w_grad   * L_grad     (edge sharpness on valid GT pixels)
          + w_egssi  * L_EG-SSI   (boundary alignment via RGB edges)
          + w_tgm    * L_TGM      (temporal consistency, no optical flow)

    When called without next-frame data, L_TGM degrades to SILog.
    When called without rgb, EG-SSI falls back to GT-gradient-based edges.

    WARNING: Requires KITTISequentialDepthDataset.
    """

    def __init__(self, w_masked=1.0, w_grad=1.0, w_egssi=1.0, w_tgm=10.0):
        super().__init__()
        self.w_masked = w_masked
        self.w_grad   = w_grad
        self.w_egssi  = w_egssi
        self.w_tgm    = w_tgm
        self.masked   = MaskedDepthLoss()
        self.grad     = GradientLoss()
        self.egssi    = EdgeGuidedSSILoss()
        self.tgm      = TemporalGradientMatchingLoss()

    def forward(self, pred, gt, mask, rgb=None,
                pred_next=None, gt_next=None, mask_next=None):
        p      = pred.clamp(EPS, MAX_DEPTH)
        l_m, _ = self.masked(p, gt, mask)
        l_g    = self.grad(p, gt, mask)
        l_e, _ = self.egssi(p, gt, mask, rgb=rgb)

        if pred_next is not None and gt_next is not None and mask_next is not None:
            _, td  = self.tgm(p, pred_next.clamp(EPS, MAX_DEPTH),
                              gt, gt_next, mask, mask_next)
            l_tgm  = td["tgm"]
        else:
            l_tgm  = torch.tensor(0., device=pred.device)

        t = (self.w_masked * l_m + self.w_grad * l_g +
             self.w_egssi  * l_e + self.w_tgm  * l_tgm)
        return t, {"total": t, "masked": l_m, "grad": l_g,
                   "egssi": l_e, "tgm": l_tgm}



def _unified_forward(criterion: nn.Module, pred: torch.Tensor,
                     gt: torch.Tensor, mask: torch.Tensor,
                     rgb: torch.Tensor = None,
                     pred_next: torch.Tensor = None,
                     gt_next: torch.Tensor = None,
                     mask_next: torch.Tensor = None):
    """
    Unified interface — normalises all 13 loss signatures to (total, dict).
    Temporal losses receive next-frame tensors when available.
    EdgeGuidedSSILoss receives raw RGB for Sobel edge detection.
    """
    if isinstance(criterion, DA3VideoCombinedLoss):
        return criterion(pred, gt, mask, rgb=rgb,
                         pred_next=pred_next, gt_next=gt_next, mask_next=mask_next)
    if isinstance(criterion, EdgeGuidedSSILoss):
        return criterion(pred, gt, mask, rgb=rgb)
    if isinstance(criterion, TemporalGradientMatchingLoss):
        if pred_next is not None:
            return criterion(pred, pred_next, gt, gt_next, mask, mask_next)
        return criterion.forward_single(pred, gt, mask)
    out = criterion(pred, gt, mask)
    if isinstance(out, tuple):
        return out
    return out, {"total": out}

# =============================================================================
# LOSS REGISTRY  -- 13 losses total (8 original + 5 new)
# =============================================================================
LOSS_REGISTRY: Dict[str, nn.Module] = {
    # ── Original 8 ──────────────────────────────────────────────────────────
    "silog":              SILogLoss(),
    "berhu":              BerHuLoss(),
    "combined_base":      CombinedBaseLoss(),       # ← original baseline
    "vnl":                VNLLoss(),
    "log_l1":             LogL1Loss(),
    "ssim_l1":            SSIMPlusL1Loss(),
    "affine_invariant":   AffineInvariantSILogLoss(),
    "robust_cauchy":      RobustCauchyLoss(),
    # ── New 5 (from DA3_Ablation_Study_v2) ──────────────────────────────────
    "masked_depth":       MaskedDepthLoss(),         # DA3  L_D
    "mae_canonical":      MAECanonicalLoss(),         # DepthPro L_MAE
    "eg_ssi":             EdgeGuidedSSILoss(),        # UniDepthV2 L_EG-SSI
    "tgm":                TemporalGradientMatchingLoss(),  # VideoDA L_TGM  [seq]
    "da3_video_combined": DA3VideoCombinedLoss(),    # Full recommended  [seq]
}
# Losses that require consecutive frame pairs (KITTISequentialDepthDataset)
TEMPORAL_LOSSES = {"tgm", "da3_video_combined"}

print("Loss functions registered:")
for k in LOSS_REGISTRY:
    tag = " <- NEW" if k in {"masked_depth", "mae_canonical",
                              "eg_ssi", "tgm", "da3_video_combined"} else ""
    print(f"  * {k}{tag}")


Loss functions registered:
  * silog
  * berhu
  * combined_base
  * vnl
  * log_l1
  * ssim_l1
  * affine_invariant
  * robust_cauchy
  * masked_depth <- NEW
  * mae_canonical <- NEW
  * eg_ssi <- NEW
  * tgm <- NEW
  * da3_video_combined <- NEW


## 5 · Metric Functions

Metrics match exactly those in `DA_3_metric_trend_analysis_FINAL.ipynb`:
- **RMSE** (pixel-level, all 4 scale variants)
- **pixel-weighted true RMSE** via `true_rmse()` — aggregates per-frame RMSEs by pixel count
- **MAE**, **AbsRel**, **SqRel**, **RMSE_log**, **log10**
- **δ<1.25**, **δ<1.25²**, **δ<1.25³**
- **Bias** (signed mean error, metres)
- All reported for: `unscaled`, `median`, `mean`, `affine` scale variants


In [10]:
def eigen_crop_mask(shape: Tuple[int, int]) -> np.ndarray:
    """Standard Eigen crop for KITTI evaluation (rows 44:396)."""
    h, w = shape[:2]
    mask = np.zeros((h, w), dtype=bool)
    top  = 44; bot = min(top + 352, h)
    mask[top:bot, 0:w] = True
    return mask


def compute_depth_errors(gt: np.ndarray, pred: np.ndarray,
                         mask: np.ndarray) -> Dict[str, float]:
    """
    KITTI standard depth metrics (masked pixels only).
    Matches compute_depth_errors() from DA_3_metric_trend_analysis_FINAL.ipynb.
    """
    gt_v   = gt[mask].astype(np.float64)
    pred_v = pred[mask].astype(np.float64)
    pred_v = np.maximum(pred_v, 1e-8)
    thresh   = np.maximum(gt_v / pred_v, pred_v / gt_v)
    rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2)))
    mae      = float(np.mean(np.abs(gt_v - pred_v)))
    abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v))
    sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / gt_v))
    rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2)))
    log10    = float(np.mean(np.abs(np.log10(gt_v) - np.log10(pred_v))))
    delta1   = float(np.mean(thresh < 1.25))
    delta2   = float(np.mean(thresh < 1.25 ** 2))
    delta3   = float(np.mean(thresh < 1.25 ** 3))
    mean_err = float(np.mean(gt_v - pred_v))    # signed bias (LiDAR - Pred)
    n_pts    = int(mask.sum())
    return dict(n_pts=n_pts, rmse=rmse, mae=mae, abs_rel=abs_rel,
                sq_rel=sq_rel, rmse_log=rmse_log, log10=log10,
                delta1=delta1, delta2=delta2, delta3=delta3,
                mean_error=mean_err,
                lidar_mean=float(np.mean(gt_v)), pred_mean=float(np.mean(pred_v)))


def compute_all_scales(gt_depth: np.ndarray,
                       pred_raw: np.ndarray,
                       mask: np.ndarray) -> Dict[str, Dict]:
    """
    Compute metrics under 4 scale-alignment strategies:
      unscaled  → raw model output
      median    → multiply pred by median(gt)/median(pred)
      mean      → multiply pred by mean(gt)/mean(pred)
      affine    → OLS: s*pred + t ≈ gt  (scale + shift)
    """
    gt_v   = gt_depth[mask]
    pred_c = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)
    results = {}

    # Unscaled
    m0 = compute_depth_errors(gt_depth, pred_c, mask)
    results["unscaled"] = {"scale": 1.0, "shift": 0.0, **m0}

    # Median scale
    s_med = float(np.median(gt_v) / np.median(pred_c[mask]))
    pred_med = np.clip(pred_c * s_med, MIN_DEPTH, MAX_DEPTH)
    m1 = compute_depth_errors(gt_depth, pred_med, mask)
    results["median"] = {"scale": s_med, "shift": 0.0, **m1}

    # Mean scale
    s_mean = float(np.mean(gt_v) / np.mean(pred_c[mask]))
    pred_mean = np.clip(pred_c * s_mean, MIN_DEPTH, MAX_DEPTH)
    m2 = compute_depth_errors(gt_depth, pred_mean, mask)
    results["mean"] = {"scale": s_mean, "shift": 0.0, **m2}

    # Affine (OLS)
    pf = pred_c[mask].ravel()
    A  = np.vstack([pf, np.ones_like(pf)]).T
    params, _, _, _ = np.linalg.lstsq(A, gt_v.ravel(), rcond=None)
    s_aff = float(params[0]); sh_aff = float(params[1])
    pred_aff = np.clip(s_aff * pred_c + sh_aff, MIN_DEPTH, MAX_DEPTH)
    m3 = compute_depth_errors(gt_depth, pred_aff, mask)
    results["affine"] = {"scale": s_aff, "shift": sh_aff, **m3}

    return results


def true_rmse(df_sub: pd.DataFrame, rmse_col: str = "rmse") -> float:
    """
    Pixel-weighted RMSE: sqrt(sum(RMSE_i² * n_pts_i) / sum(n_pts_i)).
    Correctly aggregates per-frame RMSEs — identical to true_rmse() in the
    trend analysis notebook.
    """
    valid = df_sub[[rmse_col, "n_pts"]].dropna()
    if valid.empty or valid["n_pts"].sum() == 0:
        return float("nan")
    total_sse = (valid[rmse_col] ** 2 * valid["n_pts"]).sum()
    total_pts = valid["n_pts"].sum()
    return float(np.sqrt(total_sse / total_pts))


def aggregate_results(frame_rows: List[Dict], label: str) -> Dict[str, float]:
    """
    Aggregate per-frame metric dictionaries into a single summary row.
    Reports pixel-weighted true RMSE for RMSE columns; arithmetic mean for all others.
    """
    df = pd.DataFrame(frame_rows)
    if df.empty:
        return {}

    row: Dict[str, float] = {"config": label}
    scale_variants = ["unscaled", "median", "mean", "affine"]

    for sv in scale_variants:
        for col in ["rmse", "mae", "abs_rel", "sq_rel", "rmse_log",
                    "log10", "delta1", "delta2", "delta3", "mean_error"]:
            full_col = f"{sv}_{col}"
            if full_col not in df.columns:
                continue
            if col == "rmse":
                n_col = f"{sv}_n_pts"
                if n_col in df.columns:
                    tmp = df[[full_col, n_col]].rename(columns={full_col: "rmse", n_col: "n_pts"})
                    row[full_col] = true_rmse(tmp)
                else:
                    row[full_col] = float(df[full_col].mean())
            else:
                row[full_col] = float(df[full_col].dropna().mean())

    return row


def compute_torch_metrics(pred, gt, mask) -> Dict[str, float]:
    """Per-batch metrics during training/validation (fast, torch-based)."""
    pred_c = pred.clamp(EPS, MAX_DEPTH)
    mask_b = mask.bool()
    if not mask_b.any():
        return {k: float("nan") for k in
                ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]}
    p = pred_c[mask_b].float()
    g = gt[mask_b].float().clamp(EPS)
    thresh = torch.maximum(g / p, p / g)
    return {
        "abs_rel":  float(((g - p).abs() / g).mean()),
        "sq_rel":   float(((g - p)**2 / g).mean()),
        "rmse":     float(torch.sqrt(((g - p)**2).mean())),
        "rmse_log": float(torch.sqrt((_safe_log(p) - _safe_log(g)).pow(2).mean())),
        "d1":       float((thresh < 1.25).float().mean()),
        "d2":       float((thresh < 1.25**2).float().mean()),
        "d3":       float((thresh < 1.25**3).float().mean()),
    }

print("Metric helpers ready.")


Metric helpers ready.


## 6 · Training & Validation Loops

In [11]:
def build_optimizer(model: nn.Module, optim_mode: str = "adamw") -> torch.optim.Optimizer:
    """
    Optimizer variants:
      adamw     → AdamW (original baseline, weight_decay=1e-4)
      adam      → Adam  (no weight decay, ablation)
      sgd       → SGD with Nesterov momentum (alternative classic)
    """
    trainable = list(filter(lambda p: p.requires_grad, model.parameters()))
    if optim_mode == "adam":
        return torch.optim.Adam(trainable, lr=LEARNING_RATE)
    if optim_mode == "sgd":
        return torch.optim.SGD(trainable, lr=LEARNING_RATE,
                               momentum=0.9, weight_decay=WEIGHT_DECAY,
                               nesterov=True)
    # default: adamw
    return torch.optim.AdamW(trainable, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


def build_scheduler(optimizer, steps_per_epoch: int,
                    sched_mode: str = "cosine_warmup"):
    """
    Scheduler variants:
      cosine_warmup → Cosine Annealing with linear warmup (original baseline)
      onecycle      → OneCycleLR (faster convergence)
      step          → StepLR (halve LR every N epochs, classic baseline)
    """
    from torch.optim.lr_scheduler import (
        CosineAnnealingLR, LambdaLR, OneCycleLR, StepLR, SequentialLR)

    if sched_mode == "onecycle":
        return OneCycleLR(
            optimizer, max_lr=LEARNING_RATE,
            steps_per_epoch=steps_per_epoch,
            epochs=ABLATION_EPOCHS, pct_start=0.1,
        )
    if sched_mode == "step":
        return StepLR(optimizer, step_size=max(1, ABLATION_EPOCHS//2), gamma=0.5)

    # cosine_warmup (default)
    warm_steps = WARMUP_EPOCHS * steps_per_epoch
    main_steps = (ABLATION_EPOCHS - WARMUP_EPOCHS) * steps_per_epoch
    warmup = LambdaLR(optimizer,
                      lr_lambda=lambda s: min(1.0, (s + 1) / max(warm_steps, 1)))
    cosine = CosineAnnealingLR(optimizer, T_max=max(main_steps, 1),
                               eta_min=LEARNING_RATE * 0.01)
    return SequentialLR(optimizer, [warmup, cosine], milestones=[warm_steps])


def train_one_epoch(model, loader, criterion, optimizer, scaler,
                    scheduler, device, epoch, global_step,
                    sched_mode: str = "cosine_warmup") -> Tuple[int, Dict]:
    model.model.train()
    running: Dict[str, float] = {}
    optimizer.zero_grad(set_to_none=True)

    for bi, batch in enumerate(loader):
        rgb   = batch["rgb"].to(device, non_blocking=True)
        gt    = batch["depth_gt"].to(device, non_blocking=True)
        mask  = batch["valid_mask"].to(device, non_blocking=True)
        K     = batch["K"].to(device, non_blocking=True)
        has_K = batch["has_K"].to(device, non_blocking=True)

        # Temporal next-frame tensors (present only in sequential batches)
        rgb_next   = batch.get("next_rgb")
        gt_next    = batch.get("next_depth_gt")
        mask_next  = batch.get("next_valid_mask")
        K_next     = batch.get("next_K")
        has_K_next = batch.get("next_has_K")

        with autocast(device_type="cuda",
                      enabled=MIXED_PREC and torch.cuda.is_available()):
            pred = forward_batch(model, rgb, K, has_K)

            # Predict next frame if temporal data is available
            pred_next = None
            if rgb_next is not None:
                rn  = rgb_next.to(device, non_blocking=True)
                kn  = K_next.to(device, non_blocking=True) if K_next is not None else K
                hkn = has_K_next.to(device, non_blocking=True) if has_K_next is not None else has_K
                pred_next = forward_batch(model, rn, kn, hkn)
                gt_next   = gt_next.to(device, non_blocking=True)
                mask_next = mask_next.to(device, non_blocking=True)

            total, losses = _unified_forward(
                criterion, pred, gt, mask, rgb=rgb,
                pred_next=pred_next,
                gt_next=gt_next if pred_next is not None else None,
                mask_next=mask_next if pred_next is not None else None,
            )
            total_acc    = total / ACCUM_STEPS

        scaler.scale(total_acc).backward()

        if (bi + 1) % ACCUM_STEPS == 0:
            if MAX_GRAD_NORM > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            # OneCycleLR steps every batch; cosine_warmup steps every batch too
            if scheduler and sched_mode in ("cosine_warmup", "onecycle"):
                scheduler.step()
            global_step += 1

        for k, v in losses.items():
            running[k] = running.get(k, 0.0) + float(v.detach())

    # StepLR steps every epoch
    if scheduler and sched_mode == "step":
        scheduler.step()

    avg = {k: v / max(len(loader), 1) for k, v in running.items()}
    return global_step, avg


@torch.no_grad()
def validate(model, loader, criterion, device, epoch) -> Dict[str, float]:
    model.model.eval()
    all_m: List[Dict] = []
    total_loss = 0.0
    for batch in loader:
        rgb   = batch["rgb"].to(device, non_blocking=True)
        gt    = batch["depth_gt"].to(device, non_blocking=True)
        mask  = batch["valid_mask"].to(device, non_blocking=True)
        K     = batch["K"].to(device, non_blocking=True)
        has_K = batch["has_K"].to(device, non_blocking=True)
        with autocast(device_type="cuda",
                      enabled=MIXED_PREC and torch.cuda.is_available()):
            pred    = forward_batch(model, rgb, K, has_K)
            loss, _ = _unified_forward(criterion, pred, gt, mask, rgb=rgb)
        total_loss += float(loss)
        # FIX: for affine-invariant losses, align the raw prediction to GT
        #      before computing val metrics — otherwise val_rmse measures the
        #      raw unscaled output which drifts arbitrarily during training.
        pred_for_metrics = pred
        if hasattr(criterion, "_align"):
            pred_for_metrics = criterion._align(
                pred.clamp(EPS, MAX_DEPTH), gt, mask
            )
        all_m.append(compute_torch_metrics(pred_for_metrics, gt, mask))

    def _avg(k):
        return float(np.nanmean([x[k] for x in all_m]))

    agg = {k: _avg(k) for k in ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]}
    agg["val_loss"] = total_loss / max(len(loader), 1)
    return agg

print("Training loops ready.")


Training loops ready.


## 7 · Evaluation on KITTI Test Set (all 4 scale variants)

In [12]:
def index_kitti_test(test_root: str) -> pd.DataFrame:
    rows = []
    pattern = os.path.join(test_root, "*", "*_sync",
                            "proj_depth", "groundtruth", "image_02", "*.png")
    gt_paths = glob.glob(pattern)
    if not gt_paths:
        pattern = os.path.join(test_root, "*", "*",
                               "proj_depth", "groundtruth", "image_02", "*.png")
        gt_paths = glob.glob(pattern)
    for gt_path in tqdm(gt_paths, desc="Indexing test split"):
        parts = gt_path.split(os.sep)
        try:
            gt_idx = parts.index("proj_depth")
            drive  = parts[gt_idx - 1]; date = parts[gt_idx - 2]; fname = parts[-1]
        except (ValueError, IndexError):
            continue
        rgb_path = os.path.join(test_root, date, drive, "image_02", "data", fname)
        if not os.path.exists(rgb_path):
            continue
        rows.append(dict(date=date, drive=drive, filename=fname,
                         gt_path=gt_path, rgb_path=rgb_path))
    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info("Test index: %d frames, %d drives", len(df), df["drive"].nunique())
    return df


def evaluate_model(model: nn.Module, test_df: pd.DataFrame,
                   device: torch.device,
                   config_label: str) -> List[Dict]:
    """
    Run full evaluation on the test split.
    Returns list of per-frame dicts with all 4 scale variants
    (matching the CSV schema of DA_3_metric_trend_analysis_FINAL.ipynb).
    """
    model.model.eval()
    frame_rows = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df),
                       desc=f"Eval: {config_label}"):
        try:
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            gt_depth = _read_kitti_depth(row["gt_path"])
            crop_mask = eigen_crop_mask(gt_depth.shape)
            valid_mask = ((gt_depth > MIN_DEPTH) & (gt_depth < MAX_DEPTH) & crop_mask)
            if not np.any(valid_mask):
                continue
            pred_raw = run_da3_inference(model, image_np, device)
            scale_res = compute_all_scales(gt_depth, pred_raw, valid_mask)

            frame_row = {"config": config_label}
            for sv, res in scale_res.items():
                for k, v in res.items():
                    if k not in ("scale", "shift"):
                        frame_row[f"{sv}_{k}"] = v
            frame_rows.append(frame_row)
        except Exception as e:
            log.error("Error evaluating %s: %s", row["rgb_path"], e)

    return frame_rows

print("Evaluation helpers ready.")


Evaluation helpers ready.


## 8 · Ablation Runner (HPT + Early Stopping)

Each ablation dimension now runs a **mini hyperparameter sweep** over
a coupled grid (e.g. learning rate).  The workflow per ablation label:

| Step | What happens |
|------|--------------|
| 1 | Load a **fresh** DA3METRIC-LARGE and apply the architecture variant |
| 2 | For every HPT combination: train for up to `ABLATION_EPOCHS` epochs |
| 3 | After each epoch evaluate on `val_loader`; save `temp_best_<config>__<hpt_tag>.pt` whenever `val_rmse` improves |
| 4 | Trigger **early stopping** if `val_rmse` does not improve for `EARLY_STOP_PATIENCE` consecutive epochs |
| 5 | After the sweep, pick the **winning** HPT combination (lowest `best_val_rmse`) |
| 6 | Load the winner's checkpoint; run `evaluate_model` on `test_df` **exactly once** |
| 7 | Append the final (unbiased) test metrics to `SUMMARY_CSV` |
| 8 | **Delete** all temporary checkpoint files |


In [13]:
# ── In Cell 9, before the ablation loops ──────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info("Running on: %s", device)

_base_model = load_da3_model(device)   # ONE compile call total
_base_state  = copy.deepcopy(_base_model.state_dict())

def load_da3_model_fast(device):
    """Reset weights from snapshot instead of re-loading + re-compiling."""
    _base_model.load_state_dict(copy.deepcopy(_base_state))
    return _base_model

# ══════════════════════════════════════════════════════════════════════════════
# Temp-checkpoint path helpers
# ══════════════════════════════════════════════════════════════════════════════

def _ckpt_path(ablation_label: str, hpt_tag: str) -> str:
    """Unique temp checkpoint path for one HPT trial."""
    safe = hpt_tag.replace("/", "-").replace(" ", "_")
    return os.path.join(TEMP_CKPT_DIR, f"temp_best_{ablation_label}__{safe}.pt")


def _hpt_tag(hpt_cfg: Dict) -> str:
    """Human-readable tag for one HPT combo (used in logs and filenames)."""
    return "__".join(f"{k}={v}" for k, v in sorted(hpt_cfg.items()))


# ══════════════════════════════════════════════════════════════════════════════
# Core HPT trial: train + validate + early-stop + checkpoint
# ══════════════════════════════════════════════════════════════════════════════

def _train_one_hpt_trial(
    ablation_label: str,
    hpt_cfg:        Dict,          # e.g. {"lr": 5e-5}
    loss_name:      str,
    arch_mode:      str,
    aug_mode:       str,
    optim_mode:     str,
    sched_mode:     str,
    device:         torch.device,
    train_loader:   DataLoader,
    val_loader:     DataLoader,
    seq_train_loader: DataLoader = None,   # sequential loader for temporal losses
) -> Tuple[float, str, List[Dict]]:
    """
    Train one HPT trial.

    Returns
    -------
    best_val_rmse : float   – lowest val RMSE seen during training
    ckpt_path     : str     – path to the saved temp checkpoint for that epoch
    history       : list    – per-epoch log dicts (includes hpt_tag, lr, val_*)
    """
    tag  = _hpt_tag(hpt_cfg)
    ckpt        = _ckpt_path(ablation_label, tag)
    latest_ckpt = ckpt.replace("temp_best_", "temp_latest_")

    # Resolve HPT-overridden LR (fall back to global default)
    lr = float(hpt_cfg.get("lr", LEARNING_RATE))

    log.info("  ┌─ [HPT trial] ablation=%s  tag=%s", ablation_label, tag)

    # ── Fresh model ────────────────────────────────────────────────────────
    model = load_da3_model_fast(device)
    _apply_arch(model, arch_mode)

    # ── Optimizer with HPT learning rate ──────────────────────────────────
    trainable = list(filter(lambda p: p.requires_grad, model.parameters()))
    if optim_mode == "adam":
        optimizer = torch.optim.Adam(trainable, lr=lr)
    elif optim_mode == "sgd":
        optimizer = torch.optim.SGD(
            trainable, lr=lr, momentum=0.9, weight_decay=WEIGHT_DECAY, nesterov=True)
    else:
        optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)

    # Select sequential loader for temporal losses, standard loader otherwise
    active_loader = (seq_train_loader
                     if loss_name in TEMPORAL_LOSSES and seq_train_loader is not None
                     else train_loader)

    criterion = copy.deepcopy(LOSS_REGISTRY[loss_name]).to(device)
    scheduler = build_scheduler(optimizer, len(train_loader), sched_mode)
    scaler    = GradScaler("cuda" if torch.cuda.is_available() else "cpu",
                           enabled=MIXED_PREC)

    # ── Training loop with per-epoch val evaluation + early stopping ───────
    best_val_rmse = float("inf")
    no_improve    = 0
    global_step   = 0
    start_epoch   = 1
    history: List[Dict] = []

    # ── Epoch-level resumability: reload latest checkpoint if available ────
    if os.path.exists(latest_ckpt):
        log.info("  [RESUME] Found existing state. Resuming from %s",
                 os.path.basename(latest_ckpt))
        try:
            checkpoint = torch.load(latest_ckpt, map_location=device, weights_only=False)
            model.model.load_state_dict(checkpoint['model_state'])
            optimizer.load_state_dict(checkpoint['optimizer_state'])
            if scheduler and checkpoint.get('scheduler_state'):
                scheduler.load_state_dict(checkpoint['scheduler_state'])
            scaler.load_state_dict(checkpoint['scaler_state'])
            start_epoch   = checkpoint['epoch'] + 1
            best_val_rmse = checkpoint['best_val_rmse']
            no_improve    = checkpoint['no_improve']
            global_step   = checkpoint['global_step']
            history       = checkpoint.get('history', [])
            log.info("  [RESUME] Resuming from epoch %d  best_val_rmse=%.4f",
                     start_epoch, best_val_rmse)
        except Exception as e:
            log.warning("  [RESUME] Failed to load latest ckpt (%s) — starting fresh.", e)
            best_val_rmse = float("inf")
            no_improve = 0; global_step = 0; start_epoch = 1; history = []

    for epoch in range(start_epoch, ABLATION_EPOCHS + 1):
        t0 = time.time()

        # Train
        global_step, train_losses = train_one_epoch(
            model, active_loader, criterion, optimizer,
            scaler, scheduler, device, epoch, global_step, sched_mode)

        # Validate on the held-out val slice (never on test data)
        val_m   = validate(model, val_loader, criterion, device, epoch)
        elapsed = time.time() - t0

        val_rmse    = val_m["rmse"]          # primary selection metric
        improved    = val_rmse < best_val_rmse
        improvement_marker = "  ★ IMPROVED" if improved else ""

        log.info("  │  Ep%02d/%02d  lr=%.2e  train=%.4f  val_loss=%.4f  "
                 "val_rmse=%.4f  d1=%.4f  (%.0fs)%s",
                 epoch, ABLATION_EPOCHS, lr,
                 train_losses.get("total", float("nan")),
                 val_m["val_loss"], val_rmse, val_m["d1"],
                 elapsed, improvement_marker)

        history.append({
            "config":       ablation_label,
            "hpt_tag":      tag,
            "epoch":        epoch,
            "lr":           lr,
            "train_loss":   train_losses.get("total", float("nan")),
            **{f"val_{k}": v for k, v in val_m.items()},
        })

        # Overwrite temp checkpoint whenever val_rmse improves
        if improved:
            best_val_rmse = val_rmse
            no_improve    = 0
            torch.save(
                {
                    "epoch":        epoch,
                    "model_state":  model.model.state_dict(),
                    "val_rmse":     val_rmse,
                    "hpt_cfg":      hpt_cfg,
                    "config_label": ablation_label,
                    "loss_name":    loss_name,
                    "arch_mode":    arch_mode,
                },
                ckpt,
            )
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                log.info("  │  Early stop triggered at epoch %d "
                         "(no improvement for %d epochs)",
                         epoch, EARLY_STOP_PATIENCE)
                # Save latest ckpt even on early-stop so the decision is recorded
                torch.save({
                    'epoch':           epoch,
                    'model_state':     model.model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'scheduler_state': scheduler.state_dict() if scheduler else None,
                    'scaler_state':    scaler.state_dict(),
                    'best_val_rmse':   best_val_rmse,
                    'no_improve':      no_improve,
                    'global_step':     global_step,
                    'history':         history,
                }, latest_ckpt)
                break

        # ── Save latest checkpoint (for crash/interrupt resumability) ──────
        torch.save({
            'epoch':           epoch,
            'model_state':     model.model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict() if scheduler else None,
            'scaler_state':    scaler.state_dict(),
            'best_val_rmse':   best_val_rmse,
            'no_improve':      no_improve,
            'global_step':     global_step,
            'history':         history,
        }, latest_ckpt)

    log.info("  └─ Done.  best_val_rmse=%.4f (ckpt: %s)",
             best_val_rmse, os.path.basename(ckpt))

    del model, criterion, optimizer, scaler
    torch.cuda.empty_cache()

    return best_val_rmse, ckpt, history


# ══════════════════════════════════════════════════════════════════════════════
# HPT sweep orchestrator for one ablation label
# ══════════════════════════════════════════════════════════════════════════════

def run_ablation_with_hpt(
    ablation_label:   str,
    hpt_grid:         List[Dict],         # e.g. [{"lr": 1e-4}, {"lr": 5e-5}, ...]
    loss_name:        str            = "combined_base",
    arch_mode:        str            = "decoder_only",
    aug_mode:         str            = "base",
    optim_mode:       str            = "adamw",
    sched_mode:       str            = "cosine_warmup",
    device:           torch.device   = None,
    train_loader:     DataLoader     = None,
    val_loader:       DataLoader     = None,
    test_df:          pd.DataFrame   = None,
    seq_train_loader: DataLoader     = None,   # sequential loader for temporal losses
) -> Dict[str, float]:
    """
    Full HPT-enabled ablation run for one ablation label.

    Phase 1  – Sweep: train each HPT combo with early stopping; save temp ckpt.
    Phase 2  – Select: pick the combo with lowest best_val_rmse.
    Phase 3  – Test:   load winner ckpt → run evaluate_model on test_df ONCE.
    Phase 4  – Log:    append final metrics to SUMMARY_CSV; save history CSVs.
    Phase 5  – Clean:  delete all temp checkpoints.

    Returns the test summary dict for the winning configuration.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    log.info("")
    log.info("╔" + "═" * 68 + "╗")
    log.info("║  HPT SWEEP  ablation=%-44s  ║", ablation_label)
    log.info("║  loss=%-12s arch=%-14s aug=%-8s optim=%-8s  ║",
             loss_name, arch_mode, aug_mode, optim_mode)
    log.info("║  %d trial(s) in grid × up to %d epochs each (patience=%d)%s ║",
             len(hpt_grid), ABLATION_EPOCHS, EARLY_STOP_PATIENCE,
             " " * max(0, 22 - len(str(len(hpt_grid)))))
    log.info("╚" + "═" * 68 + "╝")

    # ── Phase 1: train every HPT trial ────────────────────────────────────
    trial_results: List[Dict] = []
    all_history:   List[Dict] = []

    for hpt_cfg in hpt_grid:
        best_rmse, ckpt, history = _train_one_hpt_trial(
            ablation_label=ablation_label,
            hpt_cfg=hpt_cfg,
            loss_name=loss_name,
            arch_mode=arch_mode,
            aug_mode=aug_mode,
            optim_mode=optim_mode,
            sched_mode=sched_mode,
            device=device,
            train_loader=train_loader,
            val_loader=val_loader,
            seq_train_loader=seq_train_loader,
        )
        trial_results.append(dict(
            tag=_hpt_tag(hpt_cfg),
            hpt_cfg=hpt_cfg,
            best_val_rmse=best_rmse,
            ckpt=ckpt,
        ))
        all_history.extend(history)

    # ── Phase 2: model selection ───────────────────────────────────────────
    winner = min(trial_results, key=lambda x: x["best_val_rmse"])

    log.info("")
    log.info("  HPT Sweep Results for [%s]:", ablation_label)
    log.info("  %-40s  %s", "trial tag", "best_val_rmse")
    log.info("  " + "─" * 56)
    for t in sorted(trial_results, key=lambda x: x["best_val_rmse"]):
        marker = " ← WINNER" if t["tag"] == winner["tag"] else ""
        log.info("  %-40s  %.6f%s", t["tag"], t["best_val_rmse"], marker)

    # ── Phase 3: strict test evaluation (exactly once per ablation label) ──
    log.info("")
    log.info("  Loading winner checkpoint → test evaluation (ONCE) ...")

    model = load_da3_model_fast(device)
    _apply_arch(model, arch_mode)
    ckpt_data = torch.load(winner["ckpt"], map_location=device, weights_only=False)
    model.model.load_state_dict(ckpt_data["model_state"])
    log.info("  Winner ckpt: epoch=%d  val_rmse=%.4f",
             ckpt_data["epoch"], ckpt_data["val_rmse"])

    frame_rows = evaluate_model(model, test_df, device, ablation_label)
    summary    = aggregate_results(frame_rows, ablation_label)
    summary.update({
        "loss_fn":         loss_name,
        "arch":            arch_mode,
        "aug":             aug_mode,
        "optim":           optim_mode,
        "sched":           sched_mode,
        "winner_hpt_tag":  winner["tag"],
        "winner_val_rmse": winner["best_val_rmse"],
        "winner_epoch":    int(ckpt_data["epoch"]),
        "n_hpt_trials":    len(hpt_grid),
        "n_frames":        len(frame_rows),
    })

    # ── Phase 4: persist logs ──────────────────────────────────────────────
    # Master summary CSV (final, unbiased test numbers)
    header = not os.path.exists(SUMMARY_CSV)
    pd.DataFrame([summary]).to_csv(SUMMARY_CSV, mode="a", header=header, index=False)

    # Per-epoch training history (all HPT trials combined)
    hist_df   = pd.DataFrame(all_history)
    hist_path = os.path.join(RESULTS_DIR, f"history_{ablation_label}.csv")
    hist_df.to_csv(hist_path, index=False)

    # HPT sweep summary (one row per trial, val metrics only)
    hpt_rows = [
        {
            "ablation":      ablation_label,
            "tag":           t["tag"],
            "best_val_rmse": t["best_val_rmse"],
            "is_winner":     t["tag"] == winner["tag"],
            **t["hpt_cfg"],
        }
        for t in trial_results
    ]
    hpt_path = os.path.join(RESULTS_DIR, f"hpt_sweep_{ablation_label}.csv")
    pd.DataFrame(hpt_rows).to_csv(hpt_path, index=False)

    log.info("  ✓ SUMMARY_CSV   → %s", SUMMARY_CSV)
    log.info("  ✓ History CSV   → %s", hist_path)
    log.info("  ✓ HPT log CSV   → %s", hpt_path)

    # ── Phase 5: cleanup temp checkpoints (best + latest) ──────────────────
    for t in trial_results:
        for prefix in ["temp_best_", "temp_latest_"]:
            path = t["ckpt"].replace("temp_best_", prefix)
            try:
                os.remove(path)
                log.info("  ✓ Deleted temp ckpt: %s", os.path.basename(path))
            except FileNotFoundError:
                if prefix == "temp_best_":
                    log.warning("  ! Temp ckpt not found (no improvement ever?): %s",
                                os.path.basename(path))

    # del model
    # torch.cuda.empty_cache()
    del model
    torch.cuda.empty_cache()
    gc.collect()  # add `import gc` at top
    log.info("  ── ablation=%s complete ──", ablation_label)
    return summary


print("HPT-enabled ablation runner ready.")
print(f"  _train_one_hpt_trial  → train + early-stop + temp checkpoint")
print(f"  run_ablation_with_hpt → full sweep + model selection + test once + cleanup")


11:20:12  INFO      Running on: cuda
11:20:13  INFO      Loading depth-anything/DA3METRIC-LARGE ...


[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


11:20:13  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
11:20:13  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/depth-anything/DA3METRIC-LARGE/4010e39f3634a45bc60553321fb49fb760bd594e/config.json "HTTP/1.1 200 OK"


[INFO ] using MLP layer as FFN


11:20:16  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/model.safetensors "HTTP/1.1 302 Found"


HPT-enabled ablation runner ready.
  _train_one_hpt_trial  → train + early-stop + temp checkpoint
  run_ablation_with_hpt → full sweep + model selection + test once + cleanup


## 9 · Run All Ablations (HPT Sweep + Strict Test Isolation)

**Data split contract**
- `TRAIN_ROOT` is split 85 / 15 into a **train subset** and a **val subset** (deterministic, seed-controlled).
- `TEST_ROOT` (Eigen split) is indexed into `test_df` and used **only** inside `evaluate_model` — called strictly once per winning HPT config.
- No test-set statistics are visible during training or HPT model selection.


In [14]:
from torch.utils.data import Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info("Running on: %s", device)

# ══════════════════════════════════════════════════════════════════════════════
# Train / Val 2-way split from TRAIN_ROOT
# ══════════════════════════════════════════════════════════════════════════════
# We build the base dataset TWICE with identical parameters (same root, seed,
# max_samples) so both have the exact same _index ordering.  Subset indices are
# then applied to select either the augmented or clean version of each sample.
#
#   _full_train_aug  → augmented dataset (split="train")  → used for train subset
#   _full_train_val  → clean dataset     (split="val")    → used for val subset
# ──────────────────────────────────────────────────────────────────────────────
_full_train_aug = KITTIEigenDepthDataset(
    TRAIN_ROOT, split="train", aug_mode="base",
    augment=True, max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED)
_full_train_val = KITTIEigenDepthDataset(
    TRAIN_ROOT, split="val", aug_mode="none",
    augment=False, max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED)

assert len(_full_train_aug) == len(_full_train_val), (
    f"Dataset size mismatch: aug={len(_full_train_aug)} vs val={len(_full_train_val)}. "
    "Both must be built from the same root / seed / max_samples.")

n_total  = len(_full_train_aug)
_rng     = np.random.RandomState(VAL_SPLIT_SEED)
all_idx  = _rng.permutation(n_total)
n_val    = max(1, int(n_total * VAL_SPLIT_FRAC))
_val_idx = sorted(all_idx[:n_val].tolist())
_tr_idx  = sorted(all_idx[n_val:].tolist())

_train_subset_base = Subset(_full_train_aug, _tr_idx)   # augmented samples
_val_subset        = Subset(_full_train_val, _val_idx)  # clean samples (no aug)

# _train_loader_base = DataLoader(
#     _train_subset_base, batch_size=BATCH_SIZE, shuffle=True,
#     num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, collate_fn=collate_fn)
# _val_loader = DataLoader(
#     _val_subset, batch_size=1, shuffle=False,
#     num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
_train_loader_base = DataLoader(
    _train_subset_base, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=True,   # ← ADD True
    prefetch_factor=4,         # ← ADD 4
    collate_fn=collate_fn)

_val_loader = DataLoader(
    _val_subset, batch_size=16, shuffle=False,   # ← raise from 1
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    collate_fn=collate_fn)
# ── Test set (Eigen split) – indexed once, used only in evaluate_model ────────
test_df = index_kitti_test(TEST_ROOT)

# ── Sequential loader (for temporal losses: tgm, da3_video_combined) ─────────
# Uses the same _tr_idx so the train/val boundary is unchanged.
_full_seq_aug = KITTISequentialDepthDataset(
    TRAIN_ROOT, split="train", aug_mode="base",
    max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED)
# Subset to matched training indices where possible
_seq_available = min(len(_full_seq_aug), len(_tr_idx))
_seq_tr_idx    = [i for i in _tr_idx if i < len(_full_seq_aug)]
from torch.utils.data import Subset as _Subset
_seq_train_subset = _Subset(_full_seq_aug, _seq_tr_idx) if _seq_tr_idx else _full_seq_aug
_seq_train_loader = DataLoader(
    _seq_train_subset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=True,
    prefetch_factor=4,
    collate_fn=collate_fn_sequential)

log.info("Dataset split summary:")
log.info("  TRAIN  subset : %6d samples  (aug=%s)", len(_train_subset_base), "base")
log.info("  TRAIN  seq    : %6d pairs    (aug=base, for temporal losses)", len(_seq_train_subset))
log.info("  VAL    subset : %6d samples  (aug=none, from TRAIN_ROOT)", len(_val_subset))
log.info("  TEST   (Eigen): %6d frames   (from TEST_ROOT, never seen during HPT)", len(test_df))
log.info("  HPT grid sizes: loss=%d  arch=%d  aug=%d  optim=%d",
         len(LOSS_HPT_GRID), len(ARCH_HPT_GRID), len(AUG_HPT_GRID), len(OPTIM_HPT_GRID))

all_results: List[Dict] = []

# ── Ablation-level resumability: skip configs already in SUMMARY_CSV ─────
def get_completed_ablations() -> set:
    """Return the set of ablation_label strings already present in SUMMARY_CSV."""
    if not os.path.exists(SUMMARY_CSV):
        return set()
    try:
        df = pd.read_csv(SUMMARY_CSV)
        if "config" in df.columns:
            return set(df["config"].unique())
    except pd.errors.EmptyDataError:
        pass
    return set()

completed_ablations = get_completed_ablations()
if completed_ablations:
    log.info("[RESUME] %d ablation(s) already in SUMMARY_CSV — will skip:",
             len(completed_ablations))
    for _lbl in sorted(completed_ablations):
        log.info("  ✓ skipping: %s", _lbl)

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 1 · Loss Function Ablation
# Coupled HPT: learning rate (different losses occupy different loss landscapes)
# ══════════════════════════════════════════════════════════════════════════════
if RUN_LOSS_ABLATION:
    log.info("\n" + "─" * 70)
    log.info("GROUP 1: LOSS FUNCTION ABLATION")
    log.info("  %d loss variants × %d LR candidates = %d HPT trials each",
             len(LOSS_REGISTRY), len(LOSS_HPT_GRID),
             len(LOSS_HPT_GRID))
    log.info("─" * 70)

    for loss_name in LOSS_REGISTRY.keys():
        ablation_label = f"loss_{loss_name}"
        if ablation_label in completed_ablations:
            log.info("  [SKIP] %s already in SUMMARY_CSV", ablation_label)
            continue
        res = run_ablation_with_hpt(
            ablation_label=ablation_label,
            hpt_grid=LOSS_HPT_GRID,
            loss_name=loss_name,
            arch_mode="decoder_only",
            aug_mode="base",
            optim_mode="adamw",
            sched_mode="cosine_warmup",
            device=device,
            train_loader=_train_loader_base,
            val_loader=_val_loader,
            test_df=test_df,
            seq_train_loader=_seq_train_loader,   # auto-selected for temporal losses
        )
        all_results.append(res)

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 2 · Architecture Ablation
# Coupled HPT: learning rate (deeper unfreezing → typically lower optimal LR)
# ══════════════════════════════════════════════════════════════════════════════
if RUN_ARCH_ABLATION:
    log.info("\n" + "─" * 70)
    log.info("GROUP 2: ARCHITECTURE ABLATION")
    log.info("  3 arch variants × %d LR candidates", len(ARCH_HPT_GRID))
    log.info("─" * 70)

    for arch_mode in ["decoder_only", "last2_blocks", "full"]:
        ablation_label = f"arch_{arch_mode}"
        if ablation_label in completed_ablations:
            log.info("  [SKIP] %s already in SUMMARY_CSV", ablation_label)
            continue
        res = run_ablation_with_hpt(
            ablation_label=ablation_label,
            hpt_grid=ARCH_HPT_GRID,
            loss_name="combined_base",
            arch_mode=arch_mode,
            aug_mode="base",
            optim_mode="adamw",
            sched_mode="cosine_warmup",
            device=device,
            train_loader=_train_loader_base,
            val_loader=_val_loader,
            test_df=test_df,
        )
        all_results.append(res)

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 3 · Augmentation Ablation
# Coupled HPT: LR (aug strategy affects gradient variance → step size matters)
# Each aug mode gets its OWN train loader built from the same train-split indices
# so the val split remains identical across aug variants.
# ══════════════════════════════════════════════════════════════════════════════
if RUN_AUG_ABLATION:
    log.info("\n" + "─" * 70)
    log.info("GROUP 3: AUGMENTATION ABLATION")
    log.info("  4 aug modes × %d LR candidates  (val split identical for all)",
             len(AUG_HPT_GRID))
    log.info("─" * 70)

    for aug_mode in ["none", "flip", "base", "full"]:
        ablation_label = f"aug_{aug_mode}"
        if ablation_label in completed_ablations:
            log.info("  [SKIP] %s already in SUMMARY_CSV", ablation_label)
            continue
        # Build augmented dataset with the correct aug_mode, then apply the
        # same _tr_idx so the train/val boundary is unchanged.
        _full_aug_variant = KITTIEigenDepthDataset(
            TRAIN_ROOT, split="train", aug_mode=aug_mode,
            augment=(aug_mode != "none"), max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED)
        _aug_train_subset = Subset(_full_aug_variant, _tr_idx)
        _aug_loader = DataLoader(
            _aug_train_subset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, collate_fn=collate_fn)

        res = run_ablation_with_hpt(
            ablation_label=ablation_label,
            hpt_grid=AUG_HPT_GRID,
            loss_name="combined_base",
            arch_mode="decoder_only",
            aug_mode=aug_mode,
            optim_mode="adamw",
            sched_mode="cosine_warmup",
            device=device,
            train_loader=_aug_loader,
            val_loader=_val_loader,
            test_df=test_df,
        )
        all_results.append(res)

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 4 · Optimizer & Scheduler Ablation
# Coupled HPT: LR (optimizer + scheduler + LR are entangled)
# ══════════════════════════════════════════════════════════════════════════════
if RUN_OPTIM_ABLATION:
    log.info("\n" + "─" * 70)
    log.info("GROUP 4: OPTIMIZER / SCHEDULER ABLATION")
    log.info("  5 optim/sched combos × %d LR candidates", len(OPTIM_HPT_GRID))
    log.info("─" * 70)

    optim_sched_combos = [
        ("adamw", "cosine_warmup"),   # baseline
        ("adam",  "cosine_warmup"),
        ("sgd",   "cosine_warmup"),
        ("adamw", "onecycle"),
        ("adamw", "step"),
    ]
    for optim_mode, sched_mode in optim_sched_combos:
        ablation_label = f"optim_{optim_mode}_{sched_mode}"
        if ablation_label in completed_ablations:
            log.info("  [SKIP] %s already in SUMMARY_CSV", ablation_label)
            continue
        res = run_ablation_with_hpt(
            ablation_label=ablation_label,
            hpt_grid=OPTIM_HPT_GRID,
            loss_name="combined_base",
            arch_mode="decoder_only",
            aug_mode="base",
            optim_mode=optim_mode,
            sched_mode=sched_mode,
            device=device,
            train_loader=_train_loader_base,
            val_loader=_val_loader,
            test_df=test_df,
        )
        all_results.append(res)

# ══════════════════════════════════════════════════════════════════════════════
# Summary
# ══════════════════════════════════════════════════════════════════════════════
log.info("\n✓ All HPT-ablation sweeps complete.")
summary_df = pd.read_csv(SUMMARY_CSV)

print(f"\nTotal winning configurations evaluated on Eigen test split: {len(summary_df)}")
display_cols = [c for c in [
    "config", "unscaled_rmse", "median_rmse",
    "unscaled_abs_rel", "unscaled_delta1",
    "winner_hpt_tag", "winner_val_rmse", "winner_epoch", "n_hpt_trials",
] if c in summary_df.columns]
print(summary_df[display_cols].to_string(index=False))


11:20:17  INFO      Running on: cuda
11:20:17  INFO      [KITTI-train] 573 samples (aug=base)
11:20:17  INFO      [KITTI-val] 573 samples (aug=none)


Indexing test split:   0%|          | 0/46 [00:00<?, ?it/s]

11:20:17  INFO      Test index: 46 frames, 2 drives
11:20:18  INFO      [KITTI-Sequential-train] 570 pairs (aug=base)
11:20:18  INFO      Dataset split summary:
11:20:18  INFO        TRAIN  subset :    488 samples  (aug=base)
11:20:18  INFO        TRAIN  seq    :    485 pairs    (aug=base, for temporal losses)
11:20:18  INFO        VAL    subset :     85 samples  (aug=none, from TRAIN_ROOT)
11:20:18  INFO        TEST   (Eigen):     46 frames   (from TEST_ROOT, never seen during HPT)
11:20:18  INFO        HPT grid sizes: loss=3  arch=3  aug=3  optim=3
11:20:18  INFO      [RESUME] 15 ablation(s) already in SUMMARY_CSV — will skip:
11:20:18  INFO        ✓ skipping: arch_decoder_only
11:20:18  INFO        ✓ skipping: arch_last2_blocks
11:20:18  INFO        ✓ skipping: loss_affine_invariant
11:20:18  INFO        ✓ skipping: loss_berhu
11:20:18  INFO        ✓ skipping: loss_combined_base
11:20:18  INFO        ✓ skipping: loss_da3_video_combined
11:20:18  INFO        ✓ skipping: loss_eg_ssi
1

Eval: arch_full:   0%|          | 0/46 [00:00<?, ?it/s]

14:15:42  INFO        ✓ SUMMARY_CSV   → /STORAGE/Arka/a_MDE_Code/5_Fully_supervised_FT/super_ft_ablation_outputs/results/ablation_summary.csv
14:15:42  INFO        ✓ History CSV   → /STORAGE/Arka/a_MDE_Code/5_Fully_supervised_FT/super_ft_ablation_outputs/results/history_arch_full.csv
14:15:42  INFO        ✓ HPT log CSV   → /STORAGE/Arka/a_MDE_Code/5_Fully_supervised_FT/super_ft_ablation_outputs/results/hpt_sweep_arch_full.csv
14:15:42  INFO        ✓ Deleted temp ckpt: temp_best_arch_full__lr=0.0001.pt
14:15:42  INFO        ✓ Deleted temp ckpt: temp_latest_arch_full__lr=0.0001.pt
14:15:42  INFO        ✓ Deleted temp ckpt: temp_best_arch_full__lr=5e-05.pt
14:15:43  INFO        ✓ Deleted temp ckpt: temp_latest_arch_full__lr=5e-05.pt
14:15:43  INFO        ✓ Deleted temp ckpt: temp_best_arch_full__lr=2e-05.pt
14:15:43  INFO        ✓ Deleted temp ckpt: temp_latest_arch_full__lr=2e-05.pt
14:15:43  INFO        ── ablation=arch_full complete ──
14:15:43  INFO      
─────────────────────────────

Eval: aug_none:   0%|          | 0/46 [00:00<?, ?it/s]

... (14 outputs left out)

13:00:23  INFO        ✓ SUMMARY_CSV   → /STORAGE/Arka/a_MDE_Code/5_Fully_supervised_FT/super_ft_ablation_outputs/results/ablation_summary.csv
13:00:23  INFO        ✓ History CSV   → /STORAGE/Arka/a_MDE_Code/5_Fully_supervised_FT/super_ft_ablation_outputs/results/history_optim_adamw_onecycle.csv
13:00:23  INFO        ✓ HPT log CSV   → /STORAGE/Arka/a_MDE_Code/5_Fully_supervised_FT/super_ft_ablation_outputs/results/hpt_sweep_optim_adamw_onecycle.csv
13:00:24  INFO        ✓ Deleted temp ckpt: temp_best_optim_adamw_onecycle__lr=0.0001.pt
13:00:24  INFO        ✓ Deleted temp ckpt: temp_latest_optim_adamw_onecycle__lr=0.0001.pt
13:00:24  INFO        ✓ Deleted temp ckpt: temp_best_optim_adamw_onecycle__lr=5e-05.pt
13:00:24  INFO        ✓ Deleted temp ckpt: temp_latest_optim_adamw_onecycle__lr=5e-05.pt
13:00:24  INFO        ✓ Deleted temp ckpt: temp_best_optim_adamw_onecycle__lr=2e-05.pt
13:00:25  INFO        ✓ Deleted temp ckpt: temp_latest_optim_adamw_onecycle__lr=2e-05.pt
13:00:25  INFO    

Eval: optim_adamw_step:   0%|          | 0/46 [00:00<?, ?it/s]

15:42:15  INFO        ✓ SUMMARY_CSV   → /STORAGE/Arka/a_MDE_Code/5_Fully_supervised_FT/super_ft_ablation_outputs/results/ablation_summary.csv
15:42:15  INFO        ✓ History CSV   → /STORAGE/Arka/a_MDE_Code/5_Fully_supervised_FT/super_ft_ablation_outputs/results/history_optim_adamw_step.csv
15:42:15  INFO        ✓ HPT log CSV   → /STORAGE/Arka/a_MDE_Code/5_Fully_supervised_FT/super_ft_ablation_outputs/results/hpt_sweep_optim_adamw_step.csv
15:42:15  INFO        ✓ Deleted temp ckpt: temp_best_optim_adamw_step__lr=0.0001.pt
15:42:15  INFO        ✓ Deleted temp ckpt: temp_latest_optim_adamw_step__lr=0.0001.pt
15:42:15  INFO        ✓ Deleted temp ckpt: temp_best_optim_adamw_step__lr=5e-05.pt
15:42:15  INFO        ✓ Deleted temp ckpt: temp_latest_optim_adamw_step__lr=5e-05.pt
15:42:16  INFO        ✓ Deleted temp ckpt: temp_best_optim_adamw_step__lr=2e-05.pt
15:42:16  INFO        ✓ Deleted temp ckpt: temp_latest_optim_adamw_step__lr=2e-05.pt
15:42:16  INFO        ── ablation=optim_adamw_step


Total winning configurations evaluated on Eigen test split: 25
                   config  unscaled_rmse  median_rmse  unscaled_abs_rel  unscaled_delta1 winner_hpt_tag  winner_val_rmse  winner_epoch  n_hpt_trials
               loss_silog       2.537587     2.480774          0.073093         0.964109       lr=2e-05         1.503832             5             3
               loss_berhu       2.586817     2.489186          0.078651         0.961432       lr=5e-05         1.758120             5             3
       loss_combined_base       2.526200     2.419173          0.072493         0.967586       lr=2e-05         1.736471             4             3
                 loss_vnl       4.356280     2.928324          0.171175         0.717508       lr=5e-05         5.385635             1             3
              loss_log_l1       2.804235     2.724308          0.094919         0.948154      lr=0.0001         1.825575             3             3
             loss_ssim_l1       2.620148  

## 10 · Load & Inspect Results

In [15]:
# ── Load the summary CSV ───────────────────────────────────────────────────────
summary_df = pd.read_csv(SUMMARY_CSV)

# Define display column sets
SCALE_VARIANTS = ["unscaled", "median", "mean", "affine"]
METRIC_COLS    = ["rmse", "mae", "abs_rel", "sq_rel", "rmse_log",
                  "log10", "delta1", "delta2", "delta3", "mean_error"]

# ── Print compact summary table ────────────────────────────────────────────────
key_cols = ["config"] + [f"{sv}_rmse" for sv in SCALE_VARIANTS] + [
    "unscaled_abs_rel", "unscaled_delta1", "unscaled_delta2", "unscaled_rmse_log",
    "unscaled_mean_error"]

print("\n" + "=" * 90)
print("ABLATION SUMMARY — Pixel-Weighted True RMSE (m) and Key Metrics")
print("=" * 90)
disp = summary_df[[c for c in key_cols if c in summary_df.columns]].copy()
float_cols = [c for c in disp.columns if c != "config"]
for c in float_cols:
    disp[c] = disp[c].map(lambda x: f"{x:.4f}" if pd.notna(x) else "NaN")
print(disp.to_string(index=False))

# ── Group-level summaries ─────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("GROUPED BY ABLATION DIMENSION:")
for prefix in ["loss_", "arch_", "aug_", "optim_"]:
    sub = summary_df[summary_df["config"].str.startswith(prefix)]
    if sub.empty: continue
    print(f"\n  {prefix.upper().rstrip('_')} ablation  ({len(sub)} configs):")
    for _, r in sub.iterrows():
        uRMSE  = r.get("unscaled_rmse", float("nan"))
        mRMSE  = r.get("median_rmse",   float("nan"))
        abrel  = r.get("unscaled_abs_rel", float("nan"))
        d1     = r.get("unscaled_delta1", float("nan"))
        bias   = r.get("unscaled_mean_error", float("nan"))
        print(f"    {r['config']:<40}  "
              f"RMSE(unscaled)={uRMSE:.4f}  RMSE(median)={mRMSE:.4f}  "
              f"AbsRel={abrel:.4f}  δ<1.25={d1:.4f}  Bias={bias:+.4f}")



ABLATION SUMMARY — Pixel-Weighted True RMSE (m) and Key Metrics
                   config unscaled_rmse median_rmse mean_rmse affine_rmse unscaled_abs_rel unscaled_delta1 unscaled_delta2 unscaled_rmse_log unscaled_mean_error
               loss_silog        2.5376      2.4808    2.3670      2.3016           0.0731          0.9641          0.9961            0.1002              0.0514
               loss_berhu        2.5868      2.4892    2.3884      2.3254           0.0787          0.9614          0.9957            0.1029             -0.1651
       loss_combined_base        2.5262      2.4192    2.3428      2.3053           0.0725          0.9676          0.9960            0.0988             -0.1498
                 loss_vnl        4.3563      2.9283    2.7928      2.7506           0.1712          0.7175          0.9880            0.2069              2.5846
              loss_log_l1        2.8042      2.7243    2.5475      2.4403           0.0949          0.9482          0.9955        

## 11 · Visualisation — 15 Analysis Plots

In [16]:
# ── Colour palette ────────────────────────────────────────────────────────────
GROUP_COLORS = {
    "loss_": "#2ecc71",
    "arch_": "#3498db",
    "aug_":  "#e67e22",
    "optim_":"#9b59b6",
}
SCALE_COLORS = {
    "unscaled": "#e74c3c",
    "median":   "#27ae60",
    "mean":     "#2980b9",
    "affine":   "#8e44ad",
}

def _grp_color(label: str) -> str:
    for prefix, col in GROUP_COLORS.items():
        if label.startswith(prefix):
            return col
    return "#7f8c8d"

def _save(fig: plt.Figure, name: str, dpi: int = 140):
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, bbox_inches="tight", dpi=dpi)
    plt.close(fig)
    log.info("  ✓ %s", name)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 00 — Grand Summary Table (full metric matrix)
# ════════════════════════════════════════════════════════════════════════════════
def plot_summary_table(df: pd.DataFrame):
    cols_show = (
        ["config"] +
        [f"{sv}_{m}" for sv in SCALE_VARIANTS
                     for m in ["rmse", "abs_rel", "delta1", "mean_error"]]
    )
    cols_show = [c for c in cols_show if c in df.columns]
    disp = df[cols_show].copy()

    n_rows = len(disp) + 1   # +1 header
    n_cols = len(cols_show)
    fig_h  = max(6, 0.45 * n_rows)
    fig, ax = plt.subplots(figsize=(min(32, 2.5 * n_cols), fig_h))
    ax.axis("off")

    headers = [c.replace("_", " ") for c in cols_show]
    cell_data = [[str(v) if isinstance(v, str) else f"{v:.4f}" if pd.notna(v) else "NaN"
                  for v in row] for _, row in disp.iterrows()]

    tbl = ax.table(
        cellText=cell_data, colLabels=headers,
        cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(7.5); tbl.scale(1.0, 1.6)
    # Header style
    for j in range(n_cols):
        tbl[0, j].set_facecolor("#2c3e50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    # Row striping + colour-code by group
    for i, (_, row) in enumerate(disp.iterrows(), start=1):
        bg  = "#f0f4f8" if i % 2 == 1 else "white"
        col = _grp_color(str(row["config"]))
        for j in range(n_cols):
            tbl[i, j].set_facecolor(bg)
        tbl[i, 0].set_facecolor(col)
        tbl[i, 0].set_text_props(color="white", fontweight="bold")

    ax.set_title("Ablation Study — Full Metric Matrix (pixel-weighted true RMSE, m)",
                 fontsize=12, fontweight="bold", pad=8)
    _save(fig, "00_summary_table.png", dpi=150)

plot_summary_table(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 01 — RMSE Comparison: all 4 scale variants, all configs (bar chart)
# ════════════════════════════════════════════════════════════════════════════════
def plot_rmse_all_scales(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs))
    w = 0.2

    fig, ax = plt.subplots(figsize=(max(14, len(configs) * 0.9), 6))
    for i, (sv, col) in enumerate(SCALE_COLORS.items()):
        vals = [df.loc[df["config"] == c, f"{sv}_rmse"].values[0]
                if f"{sv}_rmse" in df.columns else float("nan")
                for c in configs]
        ax.bar(x + (i - 1.5) * w, vals, width=w, label=sv, color=col, alpha=0.85)

    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Pixel-Weighted True RMSE (m)"); ax.set_xlabel("Configuration")
    ax.set_title("RMSE Across All Scale Variants — All Ablation Configurations")
    ax.legend(title="Scale Variant")
    ax.grid(axis="y", alpha=0.4)
    # Add group separators
    for prefix in ["arch_", "aug_", "optim_"]:
        idxs = [i for i, c in enumerate(configs) if c.startswith(prefix)]
        if idxs:
            ax.axvline(x=min(idxs) - 0.5, color="grey", linestyle="--", alpha=0.5)
    _save(fig, "01_rmse_all_scales.png")

plot_rmse_all_scales(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 02 — AbsRel, SqRel, RMSE_log, log10 bar chart (unscaled)
# ════════════════════════════════════════════════════════════════════════════════
def plot_error_metrics(df: pd.DataFrame):
    metrics = ["unscaled_abs_rel", "unscaled_sq_rel", "unscaled_rmse_log", "unscaled_log10"]
    labels  = ["AbsRel", "SqRel", "RMSE_log", "log10"]
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.2

    fig, axes = plt.subplots(2, 2, figsize=(max(14, len(configs)), 10))
    axes = axes.flatten()
    for ax, metric, label in zip(axes, metrics, labels):
        vals = [df.loc[df["config"] == c, metric].values[0]
                if metric in df.columns else float("nan") for c in configs]
        colors = [_grp_color(c) for c in configs]
        bars = ax.bar(x, vals, color=colors, alpha=0.85, edgecolor="white", linewidth=0.8)
        ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45, ha="right", fontsize=7)
        ax.set_ylabel(label + " (unscaled)"); ax.set_title(label)
        ax.grid(axis="y", alpha=0.4)
        # Annotate
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=6)
    fig.suptitle("Error Metrics (Unscaled) — All Ablation Configurations",
                 fontsize=13, fontweight="bold")
    _save(fig, "02_error_metrics_unscaled.png")

plot_error_metrics(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 03 — Delta Accuracy (δ<1.25, δ<1.25², δ<1.25³) — unscaled
# ════════════════════════════════════════════════════════════════════════════════
def plot_delta_accuracy(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.25

    fig, ax = plt.subplots(figsize=(max(14, len(configs) * 0.9), 6))
    delta_info = [
        ("unscaled_delta1", "δ<1.25",   "#e74c3c"),
        ("unscaled_delta2", "δ<1.25²",  "#27ae60"),
        ("unscaled_delta3", "δ<1.25³",  "#2980b9"),
    ]
    for i, (col, lbl, col_c) in enumerate(delta_info):
        vals = [df.loc[df["config"] == c, col].values[0]
                if col in df.columns else float("nan") for c in configs]
        ax.bar(x + (i-1)*w, vals, width=w, label=lbl, color=col_c, alpha=0.85)

    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Accuracy (higher = better)"); ax.set_xlabel("Configuration")
    ax.set_title("Threshold Accuracy Metrics (δ<1.25 / δ<1.25² / δ<1.25³) — Unscaled")
    ax.legend(); ax.grid(axis="y", alpha=0.4)
    ax.set_ylim([0, 1.05])
    _save(fig, "03_delta_accuracy.png")

plot_delta_accuracy(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 04 — Bias (Mean Error, m) — signed bar chart
# ════════════════════════════════════════════════════════════════════════════════
def plot_bias(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.2

    fig, ax = plt.subplots(figsize=(max(14, len(configs)), 6))
    for i, (sv, col) in enumerate(SCALE_COLORS.items()):
        mc = f"{sv}_mean_error"
        vals = [df.loc[df["config"] == c, mc].values[0]
                if mc in df.columns else float("nan") for c in configs]
        ax.bar(x + (i-1.5)*w, vals, width=w, label=sv, color=col, alpha=0.8)

    ax.axhline(0, color="black", linewidth=1.0)
    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Mean Signed Error (LiDAR - Pred), m  (+ = model under-estimates)")
    ax.set_title("Depth Bias Across Scale Variants — All Configurations")
    ax.legend(title="Scale Variant"); ax.grid(axis="y", alpha=0.4)
    _save(fig, "04_bias_signed.png")

plot_bias(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 05 — Radar Chart: Loss Function Ablation
# ════════════════════════════════════════════════════════════════════════════════
def plot_radar_loss(df: pd.DataFrame):
    loss_df = df[df["config"].str.startswith("loss_")].copy()
    if loss_df.empty:
        return

    radar_metrics = ["unscaled_rmse", "unscaled_abs_rel", "unscaled_rmse_log",
                     "unscaled_delta1", "median_rmse", "affine_rmse"]
    radar_labels  = ["RMSE (unscaled)", "AbsRel (unscaled)", "RMSE_log (unscaled)",
                     "delta<1.25 (unscaled)", "RMSE (median)", "RMSE (affine)"]
    # Invert delta1 so all metrics are "lower = better"
    loss_df["_delta1_inv"] = 1.0 - loss_df["unscaled_delta1"]
    radar_metrics[3] = "_delta1_inv"
    radar_labels[3]  = "1-delta<1.25 (unscaled)"

    metrics_avail = [m for m in radar_metrics if m in loss_df.columns]
    labels_avail  = [l for m, l in zip(radar_metrics, radar_labels)
                     if m in loss_df.columns]

    # Normalise to [0,1]
    normed = loss_df[metrics_avail].copy()
    mn = normed.min(); mx = normed.max()
    normed = (normed - mn) / (mx - mn + 1e-9)

    N = len(metrics_avail)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(9, 9),
                           subplot_kw=dict(projection="polar"))
    colors = plt.cm.tab10(np.linspace(0, 1, len(loss_df)))

    for (_, row), color in zip(loss_df.iterrows(), colors):
        values = [normed.loc[row.name, m] for m in metrics_avail]
        values += values[:1]
        ax.plot(angles, values, color=color, linewidth=2, label=row["config"])
        ax.fill(angles, values, color=color, alpha=0.12)

    ax.set_thetagrids(np.degrees(angles[:-1]), labels_avail, fontsize=9)
    ax.set_title("Loss Function Ablation - Radar Chart (normalised, lower area = better)",
                 fontsize=11, fontweight="bold", pad=25)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)
    ax.set_ylim(0, 1)
    _save(fig, "05_radar_loss_ablation.png")

plot_radar_loss(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 06 — Radar Chart: Architecture Ablation
# ════════════════════════════════════════════════════════════════════════════════
def plot_radar_arch(df: pd.DataFrame):
    arch_df = df[df["config"].str.startswith("arch_")].copy()
    if arch_df.empty:
        return

    radar_metrics = ["unscaled_rmse", "median_rmse", "mean_rmse", "affine_rmse",
                     "unscaled_abs_rel", "unscaled_delta1"]
    radar_labels  = ["RMSE (unscaled)", "RMSE (median)", "RMSE (mean)", "RMSE (affine)",
                     "AbsRel", "δ<1.25"]
    arch_df["_delta1_inv"] = 1.0 - arch_df["unscaled_delta1"]
    radar_metrics[-1] = "_delta1_inv"; radar_labels[-1] = "1-δ<1.25"

    metrics_avail = [m for m in radar_metrics if m in arch_df.columns]
    labels_avail  = [l for m, l in zip(radar_metrics, radar_labels)
                     if m in arch_df.columns]

    normed = arch_df[metrics_avail].copy()
    mn = normed.min(); mx = normed.max()
    normed = (normed - mn) / (mx - mn + 1e-9)

    N = len(metrics_avail)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection="polar"))
    colors = ["#e74c3c", "#27ae60", "#2980b9"]

    for (_, row), color in zip(arch_df.iterrows(), colors):
        values = [normed.loc[row.name, m] for m in metrics_avail] +                  [normed.loc[row.name, metrics_avail[0]]]
        ax.plot(angles, values, color=color, linewidth=2.5, label=row["config"])
        ax.fill(angles, values, color=color, alpha=0.15)

    ax.set_thetagrids(np.degrees(angles[:-1]), labels_avail, fontsize=9)
    ax.set_title("Architecture Ablation — Radar Chart",
                 fontsize=11, fontweight="bold", pad=25)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=9)
    ax.set_ylim(0, 1)
    _save(fig, "06_radar_arch_ablation.png")

plot_radar_arch(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 07 — Heatmap: RMSE across all configs × scale variants
# ════════════════════════════════════════════════════════════════════════════════
def plot_heatmap_rmse(df: pd.DataFrame):
    rmse_cols = [f"{sv}_rmse" for sv in SCALE_VARIANTS]
    rmse_cols = [c for c in rmse_cols if c in df.columns]
    heat_df = df.set_index("config")[rmse_cols].copy()
    heat_df.columns = [c.replace("_rmse", "") for c in rmse_cols]

    fig, ax = plt.subplots(figsize=(max(8, len(SCALE_VARIANTS) * 2),
                                    max(6, len(heat_df) * 0.55)))
    sns.heatmap(heat_df.astype(float), annot=True, fmt=".3f",
                cmap="RdYlGn_r", linewidths=0.5, linecolor="grey",
                ax=ax, cbar_kws={"label": "True RMSE (m)"})
    ax.set_title("RMSE Heatmap — Configuration × Scale Variant",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Scale Variant"); ax.set_ylabel("Configuration")
    _save(fig, "07_heatmap_rmse.png")

plot_heatmap_rmse(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 08 — Heatmap: All metrics × configs (unscaled only)
# ════════════════════════════════════════════════════════════════════════════════
def plot_heatmap_all_metrics(df: pd.DataFrame):
    unscaled_cols = [f"unscaled_{m}" for m in
                     ["rmse", "mae", "abs_rel", "sq_rel", "rmse_log", "log10",
                      "delta1", "delta2", "delta3"]]
    unscaled_cols = [c for c in unscaled_cols if c in df.columns]
    heat_df = df.set_index("config")[unscaled_cols].copy()
    heat_df.columns = [c.replace("unscaled_", "") for c in unscaled_cols]

    # Normalise each column to [0, 1] so that heatmap colour is comparable
    normed = (heat_df - heat_df.min()) / (heat_df.max() - heat_df.min() + 1e-9)

    fig, ax = plt.subplots(figsize=(14, max(6, len(df) * 0.55)))
    sns.heatmap(normed, annot=heat_df.applymap(lambda x: f"{x:.3f}"),
                fmt="", cmap="RdYlGn_r", linewidths=0.5,
                ax=ax, cbar_kws={"label": "Normalised (0=best, 1=worst)"})
    ax.set_title("Metric Heatmap (Unscaled) - All Configurations. Values show raw metrics; colour shows normalised rank",
                 fontsize=11, fontweight="bold")

    ax.set_xlabel("Metric"); ax.set_ylabel("Configuration")
    _save(fig, "08_heatmap_all_metrics.png")

plot_heatmap_all_metrics(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 09 — Learning Curves (val RMSE & val Loss per epoch, all configs)
# ════════════════════════════════════════════════════════════════════════════════
def plot_learning_curves():
    hist_files = sorted(glob.glob(os.path.join(RESULTS_DIR, "history_*.csv")))
    if not hist_files:
        log.warning("No history CSVs found — skipping learning curves plot.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for hist_file in hist_files:
        try:
            h = pd.read_csv(hist_file)
            label = h["config"].iloc[0]
            color = _grp_color(label)
            if "val_rmse" in h.columns:
                axes[0].plot(h["epoch"], h["val_rmse"], label=label,
                             color=color, linewidth=1.5, alpha=0.8)
            if "val_val_loss" in h.columns or "train_loss" in h.columns:
                lc = "val_val_loss" if "val_val_loss" in h.columns else "train_loss"
                axes[1].plot(h["epoch"], h[lc], label=label,
                             color=color, linewidth=1.5, alpha=0.8)
        except Exception as e:
            log.warning("Could not load %s: %s", hist_file, e)

    for ax, title, ylabel in zip(axes,
            ["Validation RMSE (m) by Epoch", "Loss by Epoch"],
            ["Val RMSE (m)", "Loss"]):
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.legend(fontsize=6, ncol=2); ax.grid(alpha=0.4)

    fig.suptitle("Learning Curves — All Ablation Configurations",
                 fontsize=13, fontweight="bold")
    _save(fig, "09_learning_curves.png")

plot_learning_curves()


# ════════════════════════════════════════════════════════════════════════════════
# Plot 10 — Median-Scale vs Unscaled RMSE Scatter (scale-sensitivity analysis)
# ════════════════════════════════════════════════════════════════════════════════
def plot_scale_sensitivity(df: pd.DataFrame):
    if "unscaled_rmse" not in df.columns or "median_rmse" not in df.columns:
        return
    fig, ax = plt.subplots(figsize=(10, 8))
    for _, row in df.iterrows():
        color = _grp_color(str(row["config"]))
        ax.scatter(row["unscaled_rmse"], row["median_rmse"],
                   color=color, s=120, zorder=3, edgecolors="white", linewidth=0.8)
        ax.annotate(str(row["config"]).replace("loss_", "").replace("arch_", "")
                                      .replace("aug_", "").replace("optim_", ""),
                    (row["unscaled_rmse"], row["median_rmse"]),
                    fontsize=7, xytext=(5, 3), textcoords="offset points")

    vmin = min(df["unscaled_rmse"].min(), df["median_rmse"].min())
    vmax = max(df["unscaled_rmse"].max(), df["median_rmse"].max())
    ax.plot([vmin, vmax], [vmin, vmax], "k--", linewidth=1, alpha=0.5, label="y=x (no improvement)")
    ax.set_xlabel("RMSE (unscaled)"); ax.set_ylabel("RMSE (median scale aligned)")
    ax.set_title("Scale Sensitivity: Unscaled vs Median-Aligned RMSE - Points below diagonal benefit more from scale alignment")
    ax.grid(alpha=0.4); ax.legend()
    patches = [mpatches.Patch(color=c, label=p.rstrip("_"))
               for p, c in GROUP_COLORS.items()]
    ax.legend(handles=patches, title="Ablation Group",
              loc="upper left", fontsize=9)
    _save(fig, "10_scale_sensitivity.png")

plot_scale_sensitivity(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 11 — AbsRel vs δ<1.25 frontier plot (Pareto)
# ════════════════════════════════════════════════════════════════════════════════
def plot_pareto(df: pd.DataFrame):
    if "unscaled_abs_rel" not in df.columns or "unscaled_delta1" not in df.columns:
        return
    fig, ax = plt.subplots(figsize=(11, 8))
    for _, row in df.iterrows():
        color = _grp_color(str(row["config"]))
        ax.scatter(row["unscaled_abs_rel"], row["unscaled_delta1"],
                   color=color, s=130, zorder=3, edgecolors="white", linewidth=0.8)
        ax.annotate(str(row["config"]).replace("loss_", "").replace("arch_", "")
                                      .replace("aug_", "").replace("optim_", ""),
                    (row["unscaled_abs_rel"], row["unscaled_delta1"]),
                    fontsize=7, xytext=(5, 3), textcoords="offset points")

    ax.set_xlabel("AbsRel (lower = better)"); ax.set_ylabel("δ<1.25 (higher = better)")
    ax.set_title("Pareto Frontier — AbsRel vs Threshold Accuracy (Unscaled) Best configs: lower-right corner")
    ax.grid(alpha=0.4)
    patches = [mpatches.Patch(color=c, label=p.rstrip("_"))
               for p, c in GROUP_COLORS.items()]
    ax.legend(handles=patches, title="Ablation Group", fontsize=9)
    _save(fig, "11_pareto_absrel_delta1.png")

plot_pareto(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 12 — RMSE-log vs RMSE scatter (log vs metric error trade-off)
# ════════════════════════════════════════════════════════════════════════════════
def plot_rmse_vs_rmse_log(df: pd.DataFrame):
    if "unscaled_rmse" not in df.columns or "unscaled_rmse_log" not in df.columns:
        return
    fig, ax = plt.subplots(figsize=(10, 8))
    for _, row in df.iterrows():
        color = _grp_color(str(row["config"]))
        ax.scatter(row["unscaled_rmse"], row["unscaled_rmse_log"],
                   color=color, s=120, zorder=3, edgecolors="white", linewidth=0.8)
        ax.annotate(str(row["config"]).split("_", 1)[-1],
                    (row["unscaled_rmse"], row["unscaled_rmse_log"]),
                    fontsize=7, xytext=(5, 3), textcoords="offset points")

    ax.set_xlabel("RMSE (m)"); ax.set_ylabel("RMSE_log")
    ax.set_title("Metric RMSE vs Log-RMSE — Trade-off Across Configurations Best: lower-left; log-losses cluster near y-axis")
    ax.grid(alpha=0.4)
    patches = [mpatches.Patch(color=c, label=p.rstrip("_"))
               for p, c in GROUP_COLORS.items()]
    ax.legend(handles=patches, title="Ablation Group", fontsize=9)
    _save(fig, "12_rmse_vs_rmselog.png")

plot_rmse_vs_rmse_log(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 13 — Grouped Comparison: Best config per group
# ════════════════════════════════════════════════════════════════════════════════
def plot_best_per_group(df: pd.DataFrame):
    groups = {"loss_": "Loss Fn", "arch_": "Architecture",
              "aug_": "Augmentation", "optim_": "Optim/Sched"}
    best_rows = []
    for prefix, grp_label in groups.items():
        sub = df[df["config"].str.startswith(prefix)]
        if sub.empty: continue
        # Best = lowest unscaled RMSE
        best_idx = sub["unscaled_rmse"].idxmin() if "unscaled_rmse" in sub.columns else sub.index[0]
        best = sub.loc[best_idx].copy()
        best["group"] = grp_label
        best_rows.append(best)

    if not best_rows:
        return

    best_df = pd.DataFrame(best_rows)
    metrics = ["unscaled_rmse", "median_rmse", "affine_rmse",
               "unscaled_abs_rel", "unscaled_delta1"]
    labels  = ["RMSE (unscaled)", "RMSE (median)", "RMSE (affine)",
               "AbsRel", "δ<1.25"]

    x = np.arange(len(metrics)); w = 0.18
    colors = list(GROUP_COLORS.values())

    fig, ax = plt.subplots(figsize=(12, 6))
    for i, (_, row) in enumerate(best_df.iterrows()):
        vals = [row.get(m, float("nan")) for m in metrics]
        ax.bar(x + i*w, vals, width=w, label=f"{row['group']}: {row['config']}",
               color=colors[i % len(colors)], alpha=0.85)

    ax.set_xticks(x + w * (len(best_df) - 1) / 2)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("Metric Value")
    ax.set_title("Best Configuration Per Ablation Group — Key Metrics")
    ax.legend(fontsize=8, loc="upper right"); ax.grid(axis="y", alpha=0.4)
    _save(fig, "13_best_per_group.png")

plot_best_per_group(summary_df)


# ════════════════════════════════════════════════════════════════════════════════
# Plot 14 — Full Metric Comparison Table Rendered as PNG (publication-ready)
# ════════════════════════════════════════════════════════════════════════════════
def plot_full_metric_table(df: pd.DataFrame):
    """
    Renders the full numerical ablation table (8 scale×metric combos) as a PNG.
    Cells are colour-coded: green = best in column, red = worst.
    """
    report_cols = {
        "unscaled_rmse":    "RMSE-down (unscaled)",
        "median_rmse":      "RMSE-down (median)",
        "mean_rmse":        "RMSE-down (mean)",
        "affine_rmse":      "RMSE-down (affine)",
        "unscaled_abs_rel": "AbsRel↓",
        "unscaled_sq_rel":  "SqRel↓",
        "unscaled_rmse_log":"RMSE_log↓",
        "unscaled_log10":   "log10↓",
        "unscaled_delta1":  "δ<1.25↑",
        "unscaled_delta2":  "δ<1.25²↑",
        "unscaled_delta3":  "δ<1.25³↑",
        "unscaled_mean_error":"Bias↑↓",
    }
    cols_avail = {k: v for k, v in report_cols.items() if k in df.columns}
    heat_df    = df.set_index("config")[list(cols_avail.keys())].copy()
    heat_df.columns = list(cols_avail.values())

    # Determine whether lower or higher is better per column
    higher_better = {"δ<1.25↑", "δ<1.25²↑", "δ<1.25³↑"}
    annot = heat_df.applymap(lambda x: f"{x:.4f}" if pd.notna(x) else "NaN")

    # Build normalised matrix for colouring (0=best, 1=worst)
    normed = heat_df.copy().astype(float)
    for col in normed.columns:
        mn = normed[col].min(); mx = normed[col].max()
        if mx - mn < 1e-9:
            normed[col] = 0.5
        elif col in higher_better:
            normed[col] = 1.0 - (normed[col] - mn) / (mx - mn)
        else:
            normed[col] = (normed[col] - mn) / (mx - mn)

    fig, ax = plt.subplots(figsize=(max(16, len(cols_avail) * 1.6),
                                    max(6, len(df) * 0.6)))
    sns.heatmap(normed, annot=annot, fmt="", cmap="RdYlGn_r",
                linewidths=0.5, linecolor="grey", ax=ax,
                cbar_kws={"label": "0=best  1=worst (per column)"},
                vmin=0, vmax=1)
    ax.set_title("Publication-Ready Ablation Table - All Metrics & Scale Variants | Green cell = best in column | Red cell = worst in column",
        fontsize=11, fontweight="bold")
    ax.set_xlabel("Metric (↓ lower better  /  ↑ higher better)")
    ax.set_ylabel("Configuration")
    plt.xticks(rotation=30, ha="right", fontsize=8)
    _save(fig, "14_publication_table.png", dpi=180)

plot_full_metric_table(summary_df)


print("\n✓ All 15 plots saved to:", PLOT_DIR)
print("\nPlot list:")
for f in sorted(os.listdir(PLOT_DIR)):
    print(" ", f)


15:42:18  INFO        ✓ 00_summary_table.png
15:42:18  INFO        ✓ 01_rmse_all_scales.png
15:42:19  INFO        ✓ 02_error_metrics_unscaled.png
15:42:20  INFO        ✓ 03_delta_accuracy.png
15:42:20  INFO        ✓ 04_bias_signed.png
15:42:21  INFO        ✓ 05_radar_loss_ablation.png
15:42:21  INFO        ✓ 06_radar_arch_ablation.png
15:42:21  INFO        ✓ 07_heatmap_rmse.png
15:42:22  INFO        ✓ 08_heatmap_all_metrics.png
15:42:23  INFO        ✓ 09_learning_curves.png
15:42:24  INFO        ✓ 10_scale_sensitivity.png
15:42:24  INFO        ✓ 11_pareto_absrel_delta1.png
15:42:24  INFO        ✓ 12_rmse_vs_rmselog.png
15:42:25  INFO        ✓ 13_best_per_group.png
15:42:26  INFO        ✓ 14_publication_table.png



✓ All 15 plots saved to: /STORAGE/Arka/a_MDE_Code/5_Fully_supervised_FT/super_ft_ablation_outputs/plots

Plot list:
  00_summary_table.png
  01_rmse_all_scales.png
  02_error_metrics_unscaled.png
  03_delta_accuracy.png
  04_bias_signed.png
  05_radar_loss_ablation.png
  06_radar_arch_ablation.png
  07_heatmap_rmse.png
  08_heatmap_all_metrics.png
  09_learning_curves.png
  10_scale_sensitivity.png
  11_pareto_absrel_delta1.png
  12_rmse_vs_rmselog.png
  13_best_per_group.png
  14_publication_table.png


## 12 · Final Analysis & Findings

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# Final Summary Print — matches the terminal output style of
# DA_3_metric_trend_analysis_FINAL.ipynb
# ══════════════════════════════════════════════════════════════════════════════

df = pd.read_csv(SUMMARY_CSV)

print("=" * 90)
print("ABLATION STUDY FINAL RESULTS — DA3METRIC-LARGE / KITTI EIGEN-SPLIT")
print("Metrics: pixel-weighted true RMSE (same as DA_3_metric_trend_analysis_FINAL)")
print("=" * 90)

# ── Group 1: Loss Function ─────────────────────────────────────────────────────
print("\n── GROUP 1: LOSS FUNCTION ABLATION ─────────────────────────────────────")
loss_df = df[df["config"].str.startswith("loss_")].sort_values("unscaled_rmse")
print(f"{'Config':<35} {'RMSE↓':>8} {'RMSE(med)↓':>11} {'AbsRel↓':>9} "
      f"{'SqRel↓':>8} {'RMSElog↓':>10} {'δ<1.25↑':>9} {'Bias':>8}")
print("-" * 98)
for _, r in loss_df.iterrows():
    print(f"{r['config']:<35} "
          f"{r.get('unscaled_rmse', float('nan')):>8.4f} "
          f"{r.get('median_rmse', float('nan')):>11.4f} "
          f"{r.get('unscaled_abs_rel', float('nan')):>9.4f} "
          f"{r.get('unscaled_sq_rel', float('nan')):>8.4f} "
          f"{r.get('unscaled_rmse_log', float('nan')):>10.4f} "
          f"{r.get('unscaled_delta1', float('nan')):>9.4f} "
          f"{r.get('unscaled_mean_error', float('nan')):>+8.4f}")

# ── Group 2: Architecture ─────────────────────────────────────────────────────
print("\n── GROUP 2: ARCHITECTURE ABLATION ──────────────────────────────────────")
arch_df = df[df["config"].str.startswith("arch_")].sort_values("unscaled_rmse")
for _, r in arch_df.iterrows():
    print(f"{r['config']:<35} "
          f"{r.get('unscaled_rmse', float('nan')):>8.4f} "
          f"{r.get('median_rmse', float('nan')):>11.4f} "
          f"{r.get('unscaled_abs_rel', float('nan')):>9.4f} "
          f"{r.get('unscaled_sq_rel', float('nan')):>8.4f} "
          f"{r.get('unscaled_rmse_log', float('nan')):>10.4f} "
          f"{r.get('unscaled_delta1', float('nan')):>9.4f} "
          f"{r.get('unscaled_mean_error', float('nan')):>+8.4f}")

# ── Group 3: Augmentation ─────────────────────────────────────────────────────
print("\n── GROUP 3: AUGMENTATION ABLATION ──────────────────────────────────────")
aug_df = df[df["config"].str.startswith("aug_")].sort_values("unscaled_rmse")
for _, r in aug_df.iterrows():
    print(f"{r['config']:<35} "
          f"{r.get('unscaled_rmse', float('nan')):>8.4f} "
          f"{r.get('median_rmse', float('nan')):>11.4f} "
          f"{r.get('unscaled_abs_rel', float('nan')):>9.4f} "
          f"{r.get('unscaled_sq_rel', float('nan')):>8.4f} "
          f"{r.get('unscaled_rmse_log', float('nan')):>10.4f} "
          f"{r.get('unscaled_delta1', float('nan')):>9.4f} "
          f"{r.get('unscaled_mean_error', float('nan')):>+8.4f}")

# ── Group 4: Optimizer/Scheduler ─────────────────────────────────────────────
print("\n── GROUP 4: OPTIMIZER / SCHEDULER ABLATION ─────────────────────────────")
opt_df = df[df["config"].str.startswith("optim_")].sort_values("unscaled_rmse")
for _, r in opt_df.iterrows():
    print(f"{r['config']:<35} "
          f"{r.get('unscaled_rmse', float('nan')):>8.4f} "
          f"{r.get('median_rmse', float('nan')):>11.4f} "
          f"{r.get('unscaled_abs_rel', float('nan')):>9.4f} "
          f"{r.get('unscaled_sq_rel', float('nan')):>8.4f} "
          f"{r.get('unscaled_rmse_log', float('nan')):>10.4f} "
          f"{r.get('unscaled_delta1', float('nan')):>9.4f} "
          f"{r.get('unscaled_mean_error', float('nan')):>+8.4f}")

# ── Overall best ─────────────────────────────────────────────────────────────
print("\n" + "=" * 90)
if "unscaled_rmse" in df.columns:
    best_row = df.loc[df["unscaled_rmse"].idxmin()]
    print(f"\n★ BEST CONFIG (lowest unscaled RMSE): {best_row['config']}")
    print(f"  RMSE (unscaled)   = {best_row.get('unscaled_rmse', float('nan')):.4f} m")
    print(f"  RMSE (median)     = {best_row.get('median_rmse', float('nan')):.4f} m")
    print(f"  RMSE (mean)       = {best_row.get('mean_rmse', float('nan')):.4f} m")
    print(f"  RMSE (affine)     = {best_row.get('affine_rmse', float('nan')):.4f} m")
    print(f"  AbsRel            = {best_row.get('unscaled_abs_rel', float('nan')):.4f}")
    print(f"  SqRel             = {best_row.get('unscaled_sq_rel', float('nan')):.4f}")
    print(f"  RMSE_log          = {best_row.get('unscaled_rmse_log', float('nan')):.4f}")
    print(f"  log10             = {best_row.get('unscaled_log10', float('nan')):.4f}")
    print(f"  δ<1.25            = {best_row.get('unscaled_delta1', float('nan')):.4f}")
    print(f"  δ<1.25²           = {best_row.get('unscaled_delta2', float('nan')):.4f}")
    print(f"  δ<1.25³           = {best_row.get('unscaled_delta3', float('nan')):.4f}")
    print(f"  Bias (m)          = {best_row.get('unscaled_mean_error', float('nan')):+.4f}")

print("\n" + "=" * 90)
print(f"Summary CSV  → {SUMMARY_CSV}")
print(f"Plots dir    → {PLOT_DIR}")
print(f"Total configs evaluated: {len(df)}")


ABLATION STUDY FINAL RESULTS — DA3METRIC-LARGE / KITTI EIGEN-SPLIT
Metrics: pixel-weighted true RMSE (same as DA_3_metric_trend_analysis_FINAL)

── GROUP 1: LOSS FUNCTION ABLATION ─────────────────────────────────────
Config                                 RMSE↓  RMSE(med)↓   AbsRel↓   SqRel↓   RMSElog↓   δ<1.25↑     Bias
--------------------------------------------------------------------------------------------------
loss_mae_canonical                    2.4997      2.3620    0.0719   0.2108     0.0982    0.9702  +0.0317
loss_combined_base                    2.5262      2.4192    0.0725   0.2212     0.0988    0.9676  -0.1498
loss_silog                            2.5376      2.4808    0.0731   0.2107     0.1002    0.9641  +0.0514
loss_berhu                            2.5868      2.4892    0.0787   0.2270     0.1029    0.9614  -0.1651
loss_ssim_l1                          2.6201      2.4504    0.0786   0.2331     0.1047    0.9620  -0.0739
loss_masked_depth                     2.6725   